# 🧠 إطار عمل مدرّب عام (Generic Trainer Framework)

إطار تدريب **مُدار بالكامل عبر قاموس إعدادات (config)** — لا توجد داخل كود
الإطار نفسه أي أسماء أهداف، أو مخرجات، أو معاملات مكتوبة يدويًا (hardcoded).
كل ما يخص مشروعك الخاص (أسماء الأهداف، أنواعها، مخرجات نموذجك، الجداول
الزمنية للمعاملات...) تكتبه أنت في `config`، والإطار يقرأه وينفّذه.

## ما يدعمه الإطار
- عدد غير محدود من الأهداف (targets)، كل هدف بنوع مستقل تمامًا:
  - `evidential`: تقدير عدم يقين بأسلوب NIG (Normal Inverse-Gamma / Evidential Deep Learning)
  - `regression`: خسارة انحدار بسيطة (MSE / MAE / Huber) بدون عدم يقين
  - `classification`: تصنيف ثنائي أو متعدد الفئات
  - ويمكنك **تسجيل نوع جديد بالكامل** بسطرين كود دون لمس أي كلاس في الإطار.
- مدخلات مفردة أو متعددة (Single/Multi input) — الإطار لا يفتح صندوق `x` إطلاقًا،
  يمرره كما هو لنموذجك (`self.model(x)`)، فبنية مدخلاتك مسؤوليتك أنت بالكامل
  في تعريف `model_builder_fn`.
- موازنة تلقائية بين المهام المتعددة (Kendall et al., 2018) أو تعطيلها.
- أي عدد من "المعاملات القابلة للجدولة" (scheduled hyperparameters) بأي اسم
  تختاره — تُضاف/تُحذف بسطر واحد في `config['loss']['schedules']`.
- قيود منطقية/فيزيائية اختيارية بين المخرجات (مثال: `high >= low`)، قابلة
  للإضافة/الحذف دون تعديل المدرّب.

## المشكلة الأساسية التي يحلّها هذا الإطار (نظام الحفظ/الاستئناف)
في الكود القديم، عند استئناف التدريب بعد انقطاع Colab كان يتم:
1. إنشاء **مدرّب جديد** (كائن Python جديد بالكامل).
2. تحميل أوزان النموذج فقط (`load_weights`).
3. `optimizer` جديد بحالة ابتدائية صفرية (momentum = 0, variance = 0).

النتيجة: التدريب "ينسى" كل الزخم المتراكم من الحقب السابقة، رغم أن عداد
الحقبة (epoch) يستمر بشكل صحيح ظاهريًا — وهذا يُفسد فعليًا منحنى التقارب.

**الحل هنا**: نحفظ حالة **كل شيء** عبر `tf.train.Checkpoint(trainer=trainer)`
— النموذج + الـ optimizer بالكامل (بما فيه slots الزخم) + كل المعاملات
المجدولة + أوزان المهام + عداد الحقبة — ونستعيدها جميعًا معًا دفعة واحدة قبل
استئناف `fit()`. النتيجة مطابقة 100% لتدريب لم يتوقف إطلاقًا. تفاصيل الآلية
وسبب موثوقيتها موجودة في القسم المخصص لذلك، مع **اختبار تحقّق فعلي (Smoke
Test)** يمكنك تشغيله بنفسك لإثبات ذلك.

## بنية الدفتر
1. الاستيرادات
2. قاموس الإعدادات (Config) — المصدر الوحيد للحقيقة
3. سجلّ دوال الخسارة (قابل للتوسعة)
4. طبقة الموازنة التلقائية بين المهام
5. `GenericTrainer` — قلب الإطار
6. الكولباكس العامة
7. نظام الحفظ والاستئناف
8. `build_training_system` — نقطة الدخول الوحيدة
9. اختبار تحقّق فعلي (Smoke Test)
10. قالب استخدام كامل على مشروعك الحقيقي
11. تعليمات الاستئناف بعد انقطاع Colab
12. (اختياري) تدريب K-Fold
13. (اختياري) استدلال Ensemble
14. خاتمة: كيف توسّع الإطار


## 🆕 إضافات هذه النسخة (طلب تحسين محدّد)
- **`config['run']['train_mode']`**: `auto` (افتراضي، السلوك القديم) | `new` | `resume` | `warm_start`.
- **`warm_start`**: أوزان نموذج مُنجز + عقوبات الثقة/عدم اليقين تستأنف من الحقبة التي توقف عندها،
  لا من الصفر — هذا هو الإصلاح المباشر لمشكلة «الدقة لا تعود كما كانت بعد إكمال التدريب».
- **`BestModelTracker`**: مصدر واحد لتعريف «الأفضل» (بدل تعريفين متعارضين سابقًا)، حالته تُحفظ
  وتُسترجع بالكامل عبر انقطاع Colab، ويدعم `weights_snapshot: "ema_weights"` لحفظ متوسط EMA
  بدل الأوزان الخام عند كل تحسّن.
- **`force_build_trainer` → `build_trainer_variables`**: بناء متغيرات الـ optimizer بلا أي خطوة تدريب
  فعلية (كان `train_on_batch` يُحرّك `optimizer.iterations` وعزوم Adam وإحصاءات BatchNorm حتى مع
  التراجع اليدوي بعدها).
- **`TRAINER_REGISTRY`** الآن يقارن بصمة `config` (والنموذج) قبل إعادة استخدام مدرّب من نفس الجلسة.
- **`sample_weight`** مدعوم فعليًا في كل أنواع المهام (evidential/regression/classification)، ويرفض
  بصراحة إن أُعطي لنوع لا يدعمه بدل تجاهله بصمت.
- **خيارات evidential إضافية** (اختيارية بالكامل، الافتراضي = السلوك القديم تمامًا): `normalize_reg`
  (تطبيع الباقي بعرض توزيع Student-t، Meinert et al. 2023)، `beta_nll` (ترجيح الـ NLL بالتباين،
  Seitzer et al. 2022 — امتداد تجريبي لـ NIG)، `huber_weight` (خسارة Huber مباشرة على mu)،
  ودالة `nig_uncertainties_meinert` لإعادة تعريف Meinert لعدم اليقين (للتحليل/الاستدلال فقط).
- **إصلاح انهيار موازنة Kendall مع خسائر سالبة**: مهام `evidential` (NLL قد تكون سالبة) تُستثنى الآن
  افتراضياً من الموازنة التلقائية (`loss.uncertainty_weighting_exclude_task_types`)، وأُضيف مقياس
  `raw_loss`/`val_raw_loss` (بلا حدود Kendall) لاختيار أفضل نموذج — التفاصيل في ملاحظة القسم 4.
  ⚠️ نقاط حفظ التشغيلات السابقة تُستأنَف تقنياً بلا خطأ، لكن لا تستأنفها: أوزانها تدرّبت على هدف منهار
  (رؤوس NIG مفرطة الثقة) — ابدأ تشغيلاً جديداً (`train_mode="new"`).


> ### 🛠️ ملاحظة تدقيق هذه الجولة (طلب تحسين + تقييم قائمة عيوب مقترحة من نموذج آخر)
>
> **الطلبات الأربعة ونتيجة كل منها:**
> 1. **معامل بداية (استكمال/جديد)** → أُضيف `run.train_mode` بأربع قيم، بما فيها `warm_start` المخصّصة
>    لحالتك: نموذج مُنجز + استئناف الجداول من حيث توقفت.
> 2. **عقوبات الثقة تبدأ من صفر عند الاستكمال** → هذا تحديدًا ما يحله `warm_start`
>    (`run.warm_start.epochs_done`): الجداول تُضبط فورًا على قيمتها عند تلك الحقبة، لا 0.
> 3. **اعتماد الأفضل بمتوسط EMA بدل الخسارة الخام** → `BestModelTracker` يدعم الآن `smoothing: "ema"`
>    (لتنعيم *مقياس* المقارنة) و`weights_snapshot: "ema_weights"` (لحفظ *أوزان* EMA نفسها بدل الخام) —
>    خياران مستقلان لأن الطلب يحتمل كلا المعنيين.
> 4. **تقييم `nig_loss` من مدرّب آخر ودمج ميزاته** → تم ضمّ الثلاث بصفتها خيارات اختيارية لكل هدف
>    (انظر القسم 3): `normalize_reg` (Meinert et al. 2023، Eq. 11)، `beta_nll` (Seitzer et al. 2022،
>    امتداد تجريبي غير مثبت رسميًا لـ NIG)، `huber_weight`، ودالة `nig_uncertainties_meinert` لإعادة
>    تعريف Meinert لعدم اليقين (aleatoric=عرض Student-t، epistemic=1/√ν) — للاستدلال فقط، لا تدخل
>    الخسارة حتى لا تُبطل checkpoints قديمة.
>
> **تقييم قائمة العيوب المقترحة من النموذج الآخر — الأربعة صحيحة وتم إصلاحها:**
> | # | الادّعاء | التقييم | الإصلاح |
> |---|---|---|---|
> | 1 | `train_on_batch` في `force_build_trainer` له آثار جانبية حتى مع التراجع اليدوي | **صحيح** — `optimizer.iterations` يصبح 1 لا 0، عزوم Adam الداخلية (لا القابلة للتدريب) لا تُعاد، وBatchNormalization لا رجعة عن إحصاءاته المتحركة | استبدال بـ `build_trainer_variables`: تمرير أمامي بـ`training=False` + `optimizer.build()` فقط |
> | 2 | استرجاع «أفضل نسخة» بعد انقطاع Colab لا يعمل بشكل صحيح | **صحيح** — `SmoothedEarlyStopping.best_weights` في الذاكرة فقط (يضيع عند الانقطاع)، و`BestWeightsSaver.best` يبدأ من `inf` عند كل استئناف فيكتب فوق أفضل ملف بأول نتيجة تالية مهما كانت أسوأ | `BestModelTracker` يحفظ `best`/`best_epoch` في `best_meta.json` بجانب ملف الأوزان، ويُحمَّل عند البناء لا عند أول تحسّن |
> | 3 | `TRAIN_MODE` + منع إعادة استخدام `TRAINER_REGISTRY` عند تغيّر config/النموذج | **صحيح جزئيًا وأُضيف كما اقتُرح ثم وُسِّع** — الكود القديم كان يُعيد أي مدرّب مخزَّن بغضّ النظر عن config الممرَّر الآن، حتى في وضع `auto`، مما قد يُخفي تغييرًا حقيقيًا في نفس الجلسة | بصمة `config` (`config_fingerprint`) تُقارَن قبل إعادة الاستخدام؛ عدم التطابق يُعيد البناء تلقائيًا (لا يرفض بصمت) |
> | 4 | `sample_weight` غير مدعوم فعليًا | **صحيح** — `train_step`/`test_step` القديمان يفكّان `sample_weight` من الدفعة ثم لا يستخدمانه إطلاقًا في `_compute_losses` | مُرَّر الآن لكل `task_fn`، ويُرفَض بخطأ صريح لأي `task_type` لم يُسجَّل بـ`supports_sample_weight=True` |
>
> كل الإصلاحات أعلاه **مُتحقَّق منها فعليًا** (سكربتات فحص شغّلتها بنفسي، ليس تخمينًا) — تفاصيل كل فحص
> في ردّي المرافق للدفتر. الملاحظات الثلاث من التدقيق السابق (stop_gradient على المعايرة، إزالة القص
> المكرر، إعداد GPU/الأداء) ما زالت سارية دون تغيير.


## 1) الاستيرادات وإعداد البيئة

In [ ]:
# @title 1) الاستيرادات وإعداد البيئة
import os
import json
import copy
import time
import shutil
import hashlib
import tempfile
import collections.abc
import numpy as np
import tensorflow as tf
from typing import Dict, List, Optional, Tuple, Callable, Any

try:
    from google.colab import drive as _colab_drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f"TensorFlow: {tf.__version__} | Colab: {IN_COLAB}")
# ملاحظة: يعمل الإطار على Keras 2 (tf-keras) وKeras 3 معًا — اختُبر على النسختين.
# يتطلب TensorFlow 2.11+ (لتوفّر tf.keras.optimizers.AdamW وواجهة tf.train.Checkpoint الحديثة).


def mount_drive_if_needed(path: str):
    """يُركِّب Google Drive تلقائيًا إن كان أي مسار مطلوب يقع تحت /content/drive"""
    if path and str(path).startswith("/content/drive") and IN_COLAB:
        try:
            _colab_drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print(f"⚠️ تعذر تركيب Drive تلقائيًا: {e}")


def opt_variables(optimizer) -> list:
    """متغيرات الـ optimizer بغضّ النظر عن النسخة: في Keras 2 هي دالة، وفي Keras 3 خاصية (list)."""
    v = optimizer.variables
    return list(v() if callable(v) else v)


def atomic_write_json(path: str, obj: Any):
    """كتابة JSON ذرّية: نكتب لملف مؤقت ثم نستبدل — انقطاع Colab أثناء الكتابة لا يترك ملفًا مقطوعًا."""
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=float, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


## 1.1) إعداد الأداء وGPU — استغلال كامل الطاقة الحاسوبية المتاحة

هذه الخلية **جديدة** (لم تكن في النسخة السابقة). أهم نقطة توضيح أولاً:
بشكل افتراضي فإن TensorFlow يحجز عمليًا كل ذاكرة الـ GPU الحرة عند أول عملية
تخصيص — أي أن "الذاكرة الفارغة" التي تراها في `nvidia-smi` أثناء التدريب
غالبًا ليست ذاكرة غير محجوزة، بل ذاكرة محجوزة لكن **غير مُستغَلة حسابيًا**
(GPU utilization % منخفض) بسبب: حجم batch صغير نسبيًا، أو عدم استخدام
Tensor Cores (تحتاج mixed precision)، أو عنق زجاجة في تغذية البيانات
(`tf.data`). لذلك الحل الفعلي لتسريع التدريب هو:

1. **Mixed Precision (float16)** — يضاعف الإنتاجية على GPUs الحديثة
   (T4/V100/A100/L4...) عبر Tensor Cores، ويقلّل حجم الـ activations للنصف،
   مما يسمح فعليًا بزيادة `batch_size` لاستغلال المساحة المتبقية.
2. **XLA (`jit_compile`)** — يدمج (fuse) العمليات الحسابية في نواة GPU واحدة
   بدل عشرات النداءات المنفصلة، مفعّل عبر `config['optimizer']['use_xla']`.
3. **زيادة `batch_size`** — إن كانت الذاكرة فعليًا فيها متسع (جرّب مضاعفتها
   تدريجيًا حتى تقترب من حد الذاكرة، ثم تراجع خطوة واحدة للأمان). ملاحظة:
   مضاعفة `batch_size` كثيرًا قد تحتاج رفع `lr_initial` بنفس النسبة تقريبًا
   (Linear Scaling Rule) للحفاظ على جودة التقارب.
4. **تحسين `tf.data`**: `.cache()` + `.prefetch(tf.data.AUTOTUNE)` +
   `drop_remainder=True` (الأخيرة ضرورية أيضًا لضمان أشكال ثابتة مع XLA).

### ⚠️ تحذير خاص بنموذج NIG (evidential) تحديدًا
دالة الخسارة تحتوي على `tf.math.log` و`tf.math.lgamma` وقيم `nu/alpha/beta`
حسّاسة رقميًا جدًا — وهذا بالضبط نوع الحسابات التي تُصاب بـ `NaN`/`overflow`
بسهولة تحت `float16`. لذلك:
- مخرجات النموذج (`mu`, `nu`, `alpha`, `beta`, `confidence`) يجب أن تبقى
  `float32` صراحة حتى مع تفعيل mixed precision (أضف `dtype='float32'` على
  طبقات `Dense` الأخيرة في `model_builder_fn` — انظر المثال المُحدَّث في
  القسمين 9 و10 أدناه). دوال الخسارة نفسها في القسم 3 تُجري `tf.cast(..., tf.float32)`
  بالفعل، فهي آمنة بغض النظر عن سياسة الدقّة العامة.
- `USE_MIXED_PRECISION` أدناه **معطّلة افتراضيًا (`False`)** لهذا السبب —
  فعّلها فقط بعد تأكيد أن مخرجات نموذجك محمية بـ `float32` كما في الملاحظة أعلاه،
  وراقب `nig_base`/`nig_pen` بحثًا عن `NaN` في أول بضع حقب.


In [ ]:
# @title 1.1) إعداد الأداء وGPU (جديد)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            # يسمح بالتخصيص التدريجي بدل حجز كل الذاكرة دفعة واحدة —
            # يمنع تعطّل الجلسة عند مشاركة الـ GPU مع عمليات أخرى (شائع على Colab).
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(f"⚠️ تعذر ضبط memory_growth لـ {gpu.name}: {e}")
    print(f"✅ تم العثور على {len(gpus)} GPU: {[g.name for g in gpus]}")
else:
    print("⚠️ لا يوجد GPU متاح في هذه الجلسة — التدريب سيعمل على CPU (أبطأ بكثير)")

# ── Mixed Precision (float16) ──────────────────────────────────────────────
# معطّلة افتراضيًا لحساسية دالة NIG الرقمية (انظر التحذير في الخلية الشارحة أعلاه).
# فعّلها (True) بعد تطبيق dtype='float32' على مخرجات نموذجك الأخيرة.
USE_MIXED_PRECISION = False
if USE_MIXED_PRECISION and gpus:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("✅ Mixed precision (float16) مفعّل — build_optimizer سيغلّف الـ optimizer تلقائيًا بـ LossScaleOptimizer")
else:
    tf.keras.mixed_precision.set_global_policy('float32')
    print("ℹ️ Mixed precision معطّل — الحساب بالكامل float32 (الوضع الآمن الافتراضي)")

# ── XLA ─────────────────────────────────────────────────────────────────────
# لا حاجة لأي إعداد هنا: XLA يُفعَّل عبر config['optimizer']['use_xla'] الذي
# يمرَّر إلى trainer.compile(jit_compile=...) داخل build_training_system (القسم 8) —
# هذا أضمن من التفعيل العام tf.config.optimizer.set_jit(True) لأنه محصور بخطوات
# التدريب/التقييم فقط، ويسهل تعطيله لهذا المدرّب وحده إن سبّب مشاكل توافق.


## 2) قاموس الإعدادات (Config) — المصدر الوحيد للحقيقة

**فلسفة التصميم**: كل ما يخص "مهمتك" الخاصة يُعرَّف بالكامل داخل `config`. لا
يوجد داخل كود الإطار (الخلايا التالية) أي اسم هدف أو مفتاح مخرج مكتوب صراحة.
هذا يعني عمليًا:

- تغيير عدد الأهداف وأسمائها بحرية تامة، بلا حدود.
- خلط أهداف `evidential` (مع عدم يقين) مع `regression` بسيطة ومع
  `classification` في نفس النموذج وبنفس التدريب.
- إضافة/حذف جدول معامل (scheduled hyperparameter) بإضافة/حذف سطر واحد في
  `config['loss']['schedules']` دون لمس أي كلاس.
- إضافة نوع مهمة جديد بالكامل (`task_type`) عبر `register_task_type(...)`
  (القسم التالي) دون تعديل `GenericTrainer` إطلاقًا.

`DEFAULT_CONFIG` أدناه هو مجرد **هيكل افتراضي** (قيم منطقية آمنة + توثيق لكل
مفتاح) — أنت تكتب `config` مشروعك الخاص وتمرره لـ `build_config()` الذي يدمجه
فوق الافتراضي.

In [ ]:
# @title 2) قاموس الإعدادات الافتراضي + دالة الدمج
VALID_TRAIN_MODES = ("auto", "new", "resume", "warm_start")

DEFAULT_CONFIG: Dict[str, Any] = {
    # ─────────────────────────────────────────────────────────────
    # إعدادات التشغيل العامة
    # ─────────────────────────────────────────────────────────────
    "run": {
        "run_dir": "/content/drive/MyDrive/training_runs/default_run",  # المسار الدائم (يُفضّل Drive)
        "mirror_dir": None,      # مسار احتياطي اختياري (مثلاً Drive) إن كان run_dir محليًا سريعًا
        "mirror_every": 1,       # كل كم حقبة يتم نسخ run_dir بالكامل إلى mirror_dir
        "epochs": 40,
        "batch_size": 64,
        "seed": 42,
        "verbose": 1,

        # ── طريقة البدء (جديد) ───────────────────────────────────────────────
        #  "auto"       : يستأنف من آخر checkpoint إن وُجد، وإلا يبدأ من الصفر (السلوك القديم)
        #  "new"        : يبدأ من الصفر دائمًا. أي حالة سابقة في run_dir تُنقل إلى _archive/ (لا تُحذف)
        #  "resume"     : يستأنف الحالة الكاملة (أوزان + optimizer + جداول). يفشل بوضوح إن لم يجد checkpoint
        #  "warm_start" : أوزان نموذج موجود + optimizer جديد، وتستمر الجداول من الحقبة run.warm_start.epochs_done
        "train_mode": "auto",
        "on_existing": "archive",   # عند new/warm_start وفي run_dir حالة سابقة: "archive" (انقلها) | "error" (توقّف)
        "warm_start": {
            "weights_path": None,        # ملف أوزان النموذج الأساسي (.weights.h5) — إلزامي في وضع warm_start
            "epochs_done": None,         # كم حقبة تدرّبها هذا النموذج فعلًا — إلزامي (يحدد موضع الجداول)
            "lr_rewarmup_epochs": 0,     # تسخين قصير لمعدل التعلم لأن optimizer جديد (0 = استمرار الجدول كما هو)
        },
        "strict_epoch_guard": True,   # يفشل fit() إن كان initial_epoch لا يطابق عدد الحقب المنجزة فعلًا
    },

    # ─────────────────────────────────────────────────────────────
    # الأهداف (targets) — عرّف هنا كل مخرج تريد تدريب النموذج عليه.
    # كل هدف قاموس مستقل بالكامل عن غيره. مثال (احذفه/بدّله بمشروعك):
    #
    # "targets": {
    #     "my_target": {
    #         "true_key": "y_my_target",        # المفتاح داخل y الحقيقي (dict)
    #         "task_type": "evidential",         # evidential | regression | classification | (نوعك الخاص)
    #         "output_keys": {                   # مطابقة أسماء مخرجات نموذجك (dict outputs)
    #             "mu": "y_my_target", "nu": "y_my_target_nu",
    #             "alpha": "y_my_target_alpha", "beta": "y_my_target_beta",
    #             "confidence": "y_my_target_confidence",
    #         },
    #         "loss_weight": 1.0,                # وزن ابتدائي (قابل للتحديث التلقائي لاحقًا)
    #         "use_calibration_loss": True,       # فقط لـ evidential
    #         "lambda_reg_var": "lambda_reg",     # اسم المتغير المجدول المستخدم في هذا الهدف
    #         "lambda_calib_var": "lambda_calib",
    #
    #         # ── خيارات evidential الإضافية (كلها افتراضيًا = السلوك القديم تمامًا) ──
    #         "normalize_reg": False,             # Meinert 2023 Eq.(11): قسمة الباقي على w_St داخل منظِّم الأدلة
    #         "normalize_reg_stop_grad": True,    # قطع التدرج عن w_St في المنظِّم (اختيار تصميم — الورقة لا تحدده)
    #         "beta_nll": 0.0,                    # Seitzer 2022: وزن NLL بـ (w_St²)^beta_nll — جرّب 0.5 (تجريبي على NIG)
    #         "huber_weight": 0.0,                # خسارة Huber مباشرة على mu كي لا "يفسّر" عدم اليقين الخطأ
    #         "huber_delta": 1.0,                 # ⚠️ يجب أن يناسب مقياس هدفك (بعد التطبيع)
    #         "alpha_floor_straight_through": False,  # مرّر التدرج عبر قصّ alpha<1 بدل قتله
    #     },
    # },
    "targets": {},

    # ─────────────────────────────────────────────────────────────
    # الخسارة (loss)
    # ─────────────────────────────────────────────────────────────
    "loss": {
        "use_uncertainty_weighting": True,   # موازنة تلقائية بين المهام (Kendall et al.)
        # أنواع مهام تُستثنى من موازنة Kendall وتُجمَع بوزنها الثابت (loss_weight). خسارة evidential
        # هي NLL كاملة قد تكون سالبة، ومعها تنحدر 0.5·exp(-s)·L + 0.5·s بلا حدّ (s → -∞) فتنفجر الخسارة
        # الكلية نحو -∞ — راجع ملاحظة القسم 4. أفرِغ القائمة ([]) فقط إن كنت متأكداً أن كل خسائرك موجبة.
        "uncertainty_weighting_exclude_task_types": ["evidential"],
        "schedules": {
            # اسم تختاره أنت: {"start":.., "end":.., "warmup_epochs":.., "schedule": "linear|cosine|exponential|sqrt"}
            # "lambda_reg":   {"start": 0.0, "end": 0.0, "warmup_epochs": 20, "schedule": "linear"},
            # "lambda_calib": {"start": 0.0, "end": 0.0, "warmup_epochs": 20, "schedule": "cosine"},
        },
        "constraints": [
            # قيود منطقية/فيزيائية اختيارية بين المخرجات، مثال:
            # {"name": "high_low_order", "fn": my_constraint_fn, "weight_var": "penalty_weight"},
        ],
    },

    # ─────────────────────────────────────────────────────────────
    # المُحسِّن (optimizer)
    # ─────────────────────────────────────────────────────────────
    "optimizer": {
        "name": "adamw",              # adamw | adam
        "lr_initial": 5e-5,
        "lr_min": 5e-7,
        "lr_warmup_epochs": 3,
        "lr_schedule": {"type": "cosine_restarts", "cycle_length": 10, "cycle_mult": 1.5},
        # أنواع lr_schedule المتاحة: "constant" | "cosine" (مرة واحدة حتى total_epochs) | "cosine_restarts"
        "weight_decay": 1e-4,
        "clip_norm": 1.0,
        "use_ema": True,
        "ema_momentum": 0.999,
        "use_xla": False,   # فعّلها True لتسريع GPU عبر XLA (جرّبها أولاً على مشروعك)
    },

    # ─────────────────────────────────────────────────────────────
    # الكولباكس (callbacks)
    # ─────────────────────────────────────────────────────────────
    "callbacks": {
        # مصدر واحد لـ"أفضل نسخة" + التوقف المبكر (BestModelTracker):
        "early_stopping": {
            "monitor": "val_loss",      # أي مقياس يظهر في logs (مثل val_a_mae أو val_b_accuracy)
            "mode": "min",              # "min" للخسارة/الخطأ، "max" للدقة
            "patience": 15,
            "min_delta": 1e-4,
            "smoothing": "window",      # طريقة تنعيم *المقياس* قبل اختيار الأفضل: "none" | "window" | "ema"
            "smoothing_window": 3,      # لـ "window": متوسط آخر N حقب
            "ema_beta": 0.7,            # لـ "ema": s_t = beta*s_(t-1) + (1-beta)*x_t  (≈ نافذة 1/(1-beta) حقبة)
            "restore_best_weights": True,
            # أي *أوزان* تُحفَظ عند كل تحسّن: "raw" (أوزان تلك الحقبة كما هي) | "ema_weights" (متوسط EMA
            # المتراكم داخل optimizer عبر optimizer['use_ema'] — أكثر استقرارًا، لكنه يتطلب use_ema=True).
            # هذا مستقل تمامًا عن "smoothing" أعلاه (ذاك ينعّم *مقياس المقارنة*، هذا يغيّر *ماذا نحفظ*).
            "weights_snapshot": "raw",
        },
        "task_weight_update_frequency": 0,   # >0 لتفعيل تحديث يدوي (فقط عند تعطيل uncertainty_weighting)
        "metrics_log_every": 5,
        "snapshots": {"epochs": [], "dir": "snapshots"},   # اتركها [] لتعطيل حفظ اللقطات الإضافية
    },

    # ─────────────────────────────────────────────────────────────
    # نظام الحفظ/الاستئناف (checkpoint)
    # ─────────────────────────────────────────────────────────────
    "checkpoint": {
        "save_every": 1,             # احفظ كل epoch (الوحدة الدنيا لضمان الاستئناف)
        "max_to_keep": 3,
        "save_best_weights": True,   # ملف best.weights.h5 لأفضل أوزان (يُحفظ ذرّيًا ويبقى صحيحًا عبر الاستئناف)
    },
}


def deep_update(base: dict, override: dict) -> dict:
    """دمج عميق: override تُكتب فوق base دون فقدان أي مفتاح غير مذكور في override"""
    result = copy.deepcopy(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(result.get(k), dict):
            result[k] = deep_update(result[k], v)
        else:
            result[k] = v
    return result


def build_config(user_config: dict) -> dict:
    """يدمج إعدادات المستخدم فوق إعدادات الإطار الافتراضية، ويتحقق من الحد الأدنى اللازم"""
    cfg = deep_update(DEFAULT_CONFIG, user_config)
    if not cfg["targets"]:
        raise ValueError("❌ يجب تعريف هدف واحد على الأقل داخل config['targets']")
    for name, tcfg in cfg["targets"].items():
        for required in ("true_key", "task_type", "output_keys"):
            if required not in tcfg:
                raise ValueError(f"❌ الهدف '{name}': المفتاح المطلوب '{required}' غير موجود")

    run = cfg["run"]
    if str(run["train_mode"]).lower() not in VALID_TRAIN_MODES:
        raise ValueError(f"❌ run.train_mode='{run['train_mode']}' غير صالح — اختر واحدًا من {VALID_TRAIN_MODES}")
    run["train_mode"] = str(run["train_mode"]).lower()
    if run["on_existing"] not in ("archive", "error"):
        raise ValueError("❌ run.on_existing يجب أن يكون 'archive' أو 'error'")

    es = cfg["callbacks"]["early_stopping"]
    if es["smoothing"] not in ("none", "window", "ema"):
        raise ValueError("❌ early_stopping.smoothing يجب أن يكون 'none' أو 'window' أو 'ema'")
    if es["mode"] not in ("min", "max"):
        raise ValueError("❌ early_stopping.mode يجب أن يكون 'min' أو 'max'")
    if not 0.0 <= float(es["ema_beta"]) < 1.0:
        raise ValueError("❌ early_stopping.ema_beta يجب أن يكون في [0, 1)")
    return cfg


# ── بصمة الإعدادات: تُستخدم لمنع إعادة استخدام مدرّب قديم في الذاكرة بعد تغيير config أو النموذج ──
# مفاتيح run التي لا تغيّر «هوية» المدرّب (تغييرها مشروع أثناء الجلسة: رفع epochs مثلًا)
_FINGERPRINT_IGNORED_RUN_KEYS = ("epochs", "verbose", "batch_size", "seed", "train_mode", "on_existing", "warm_start")


def _canonical(o):
    if isinstance(o, dict):
        return {str(k): _canonical(v) for k, v in sorted(o.items(), key=lambda kv: str(kv[0]))}
    if isinstance(o, (list, tuple)):
        return [_canonical(v) for v in o]
    if callable(o):
        return f"<fn:{getattr(o, '__module__', '')}.{getattr(o, '__qualname__', repr(o))}>"
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o if isinstance(o, (int, float, str, bool, type(None))) else repr(o)


def config_fingerprint(cfg: dict) -> str:
    c = {k: v for k, v in cfg.items() if k != "run"}
    c["run"] = {k: v for k, v in cfg["run"].items() if k not in _FINGERPRINT_IGNORED_RUN_KEYS}
    return hashlib.sha256(json.dumps(_canonical(c), sort_keys=True).encode("utf-8")).hexdigest()[:16]


def model_signature(model: tf.keras.Model) -> list:
    """توقيع بنيوي للنموذج: نوع كل طبقة + دالة التنشيط + أشكال الأوزان (لا نستخدم أسماء الطبقات لأن Keras يُرقّمها تلقائيًا)"""
    sig = []
    for layer in model.layers:
        act = getattr(layer, "activation", None)
        shapes = tuple(tuple(int(s) for s in w.shape) for w in layer.weights)
        sig.append((type(layer).__name__, getattr(act, "__name__", None), shapes))
    return sig


## 3) سجلّ دوال الخسارة (Loss Registry) — قابل للتوسعة بالكامل

كل "نوع مهمة" (`task_type`) هو ببساطة:

```
دالة بتوقيع  (y_true, outputs, target_cfg, scheduled_vars) -> (loss_scalar, stats_dict)
+ قائمة بأسماء الإحصائيات (stat_keys) التي تُرجعها الدالة دائمًا (نفس المفاتيح
  في كل استدعاء، بلا شرط) — تُستخدم لإنشاء metrics التتبّع تلقائيًا عند بناء المدرّب.
```

**لإضافة نوع مهمة جديد بالكامل** (مثلاً quantile regression):
```python
def quantile_task_loss(y_true, outputs, target_cfg, scheduled_vars):
    ...
    return loss, {"pinball": ...}

register_task_type("quantile", quantile_task_loss, ["pinball"])
```
ثم استخدم `"task_type": "quantile"` في `config['targets'][...]` — لن تحتاج
لتعديل `GenericTrainer` إطلاقًا.

## 3) سجلّ دوال الخسارة (Loss Registry) — قابل للتوسعة بالكامل

كل "نوع مهمة" (`task_type`) هو ببساطة:

```
دالة بتوقيع  (y_true, outputs, target_cfg, scheduled_vars) -> (loss_scalar, stats_dict)
+ قائمة بأسماء الإحصائيات (stat_keys) التي تُرجعها الدالة دائمًا (نفس المفاتيح
  في كل استدعاء، بلا شرط) — تُستخدم لإنشاء metrics التتبّع تلقائيًا عند بناء المدرّب.
```

**لإضافة نوع مهمة جديد بالكامل** (مثلاً quantile regression):
```python
def quantile_task_loss(y_true, outputs, target_cfg, scheduled_vars):
    ...
    return loss, {"pinball": ...}

register_task_type("quantile", quantile_task_loss, ["pinball"])
```
ثم استخدم `"task_type": "quantile"` في `config['targets'][...]` — لن تحتاج
لتعديل `GenericTrainer` إطلاقًا.

### 🆕 خيارات evidential إضافية (اختيارية، الافتراضي = السلوك القديم)
تُضبط داخل `config['targets']['<اسم الهدف>']`:
- `normalize_reg: True` — يقسم الباقي في منظِّم الأدلة على عرض توزيع Student-t (`w_St`) بدل استخدامه خامًا
  (Meinert et al. 2023، معادلة 11). يقلّل تسرّب عدم اليقين الإحصائي (aleatoric) إلى تقدير عدم اليقين
  المعرفي (epistemic) عندما تتفاوت شدة الضجيج عبر العينات — وهذا بالضبط نمط بيانات الأسعار.
- `beta_nll: 0.5` — يرجّح NLL بـ `w_St^(2·beta_nll)` (امتداد لفكرة Seitzer et al. 2022 المصمَّمة أصلًا
  لـ Gaussian NLL؛ تطبيقها على NIG هنا اجتهاد غير موثَّق في الأدبيات — قيّمها تجريبيًا).
- `huber_weight: 0.1` — يضيف خسارة Huber مباشرة على `mu` فلا يستطيع النموذج «تفسير» خطأ التوقع بتضخيم
  عدم اليقين بدل تحسين الدقة.
- `alpha_floor_straight_through: True` — القصّ القديم `max(alpha, 1.0)` يقتل التدرّج تمامًا حين تكون
  `alpha<1` (شائع مع رأس `softplus` وحده). هذا الخيار يُبقي القيمة الأمامية كما هي لكنه يُمرّر التدرّج
  (straight-through estimator) بدل قتله — بديل أنظف هو تغيير رأس النموذج نفسه إلى `1+softplus(x)`.
- دالة مستقلة `nig_uncertainties_meinert(nu, alpha, beta)` تُعيد تعريف Meinert لعدم اليقين
  (aleatoric=عرض Student-t، epistemic=1/√ν) — **للاستدلال والتحليل فقط**، لم تُدمَج في الخسارة أو في
  إحصاءات التدريب المحفوظة، حتى تبقى checkpoints ونماذج قديمة صالحة دون تغيير في معناها.


In [ ]:
# @title 3) سجلّ دوال الخسارة (Loss Registry)
TASK_REGISTRY: Dict[str, Dict[str, Any]] = {}


def register_task_type(name: str, fn: Callable, stat_keys: List[str], supports_sample_weight: bool = False):
    """سجّل نوع مهمة جديد.
    fn: (y_true, outputs, target_cfg, scheduled_vars) -> (loss, stats_dict)
    إن أعلنتَ supports_sample_weight=True فيجب أن تقبل fn وسيطًا إضافيًا sample_weight=None
    (وإلا فإعطاء sample_weight لهدف من هذا النوع يرفع خطأً صريحًا بدل أن يُتجاهل بصمت)."""
    TASK_REGISTRY[name] = {"fn": fn, "stat_keys": list(stat_keys),
                           "supports_sample_weight": bool(supports_sample_weight)}


# ───────────────────────────── مساعدات الأوزان ─────────────────────────────

def _flat(x):
    return tf.reshape(tf.cast(x, tf.float32), [-1])


def _wmean(x, w=None):
    """متوسط مرجَّح Σ(w·x)/Σw — لا يعتمد على مقياس الأوزان، ويُرجع 0 (لا NaN) إن كانت كلها أصفارًا.
    مع w=None يساوي reduce_mean تمامًا."""
    x = tf.cast(x, tf.float32)
    if w is None:
        return tf.reduce_mean(x)
    w = tf.cast(w, tf.float32)
    return tf.math.divide_no_nan(tf.reduce_sum(x * w), tf.reduce_sum(w))


def _wstd(x, w=None):
    if w is None:
        return tf.math.reduce_std(x)
    m = _wmean(x, w)
    return tf.sqrt(_wmean(tf.square(x - m), w) + 1e-12)


# ───────────────────────────── evidential (NIG) ─────────────────────────────

def compute_confidence_calibration_loss(y_true, y_pred, confidence, sample_weight=None):
    y_true = _flat(y_true)
    y_pred = _flat(y_pred)
    confidence = _flat(confidence)
    w = None if sample_weight is None else _flat(sample_weight)

    # قطع تدرّج y_pred يمنع calibration loss من التأثير مباشرة على mu (Bug #2 الموثّق سابقًا)
    y_pred_detached = tf.stop_gradient(y_pred)
    abs_error = tf.abs(y_true - y_pred_detached)
    batch_mean = _wmean(abs_error, w)
    batch_std = _wstd(abs_error, w)
    max_expected_error = tf.stop_gradient(batch_mean + 1.5 * batch_std)
    normalized_error = tf.clip_by_value(abs_error / (max_expected_error + 1e-8), 0.0, 1.0)
    target_confidence = 1.0 - normalized_error

    diff = confidence - target_confidence
    focal_weight = tf.pow(tf.abs(diff), 2.0)
    focal_calibration_loss = _wmean(focal_weight * tf.square(diff), w)

    conf_std = _wstd(confidence, w)
    diversity_penalty = tf.maximum(0.0, 0.15 - conf_std)
    saturation_penalty = _wmean(
        tf.maximum(0.0, confidence - 0.95) + tf.maximum(0.0, 0.05 - confidence), w
    )
    return focal_calibration_loss + 0.5 * diversity_penalty + 0.3 * saturation_penalty


def nig_w_st(nu, alpha, beta):
    """عرض توزيع Student-t الناتج عن NIG: w_St = sqrt(beta(1+nu)/(nu*alpha))  (Meinert et al. 2023, Eq. 9)"""
    return tf.sqrt(beta * (1.0 + nu) / (nu * alpha))


def nig_regularizer(error, nu, alpha, beta, normalize=False, stop_grad=True):
    """منظِّم الأدلة:
       normalize=False → |y-γ|·(2ν+α)          (Amini et al. 2020 — السلوك القديم)
       normalize=True  → |y-γ|/w_St ·(2ν+α)    (Meinert et al. 2023, Eq. 11 عند p=1)
    stop_grad: قطع التدرج عن w_St. الورقة لا تحدد ذلك — هذا اختيار تصميم كي لا يستطيع النموذج
    تصغير المنظِّم بتضخيم w_St وحده."""
    resid = tf.abs(error)
    if normalize:
        w_st = nig_w_st(nu, alpha, beta)
        if stop_grad:
            w_st = tf.stop_gradient(w_st)
        resid = resid / (w_st + 1e-8)
    return resid * (2.0 * nu + alpha)


def nig_uncertainties_meinert(nu, alpha, beta) -> Dict[str, tf.Tensor]:
    """إعادة تعريف Meinert et al. 2023 (Eq. 10): aleatoric = w_St (بوحدات y)، epistemic = 1/sqrt(nu) (بلا وحدات).
    للاستدلال/التحليل فقط — لا تدخل في الخسارة ولا في metrics التدريب (كي تبقى checkpoints القديمة صالحة)."""
    nu = tf.maximum(tf.cast(nu, tf.float32), 1e-6)
    alpha = tf.maximum(tf.cast(alpha, tf.float32), 1e-6)
    beta = tf.cast(beta, tf.float32)
    return {"aleatoric_wst": nig_w_st(nu, alpha, beta), "epistemic_inv_sqrt_nu": 1.0 / tf.sqrt(nu)}


def _floor_alpha(alpha, target_cfg):
    """alpha<1 مستحيل في NIG السليم (E[σ²]=β/(α-1)). القصّ عند 1.0 هو السلوك القديم لكنه يقتل التدرج
    عن alpha عندما يكون <1 (يحدث كثيرًا إن كان رأس alpha هو softplus وحده). البديل الأفضل هو 1+softplus داخل
    النموذج (Meinert Eq. 6). خيار straight-through هنا يُبقي القيمة الأمامية كما هي ويمرّر التدرج."""
    clamped = tf.maximum(alpha, 1.0)
    if target_cfg.get("alpha_floor_straight_through", False):
        return alpha + tf.stop_gradient(clamped - alpha)
    return clamped


def evidential_task_loss(y_true, outputs, target_cfg, scheduled_vars, sample_weight=None):
    keys = target_cfg["output_keys"]
    w = None if sample_weight is None else _flat(sample_weight)
    y_true = _flat(y_true)
    gamma = _flat(outputs[keys["mu"]])
    nu = _flat(outputs[keys["nu"]])
    alpha = _flat(outputs[keys["alpha"]])
    beta = _flat(outputs[keys["beta"]])

    error = y_true - gamma
    omega = tf.maximum(2.0 * beta * (1.0 + nu), 1e-6)
    nu = tf.maximum(nu, 1e-6)
    alpha = _floor_alpha(alpha, target_cfg)

    nig_base = (
        0.5 * tf.math.log(np.pi / nu)
        - alpha * tf.math.log(omega)
        + (alpha + 0.5) * tf.math.log(nu * tf.square(error) + omega)
        + tf.math.lgamma(alpha) - tf.math.lgamma(alpha + 0.5)
    )
    nig_pen = nig_regularizer(
        error, nu, alpha, beta,
        normalize=bool(target_cfg.get("normalize_reg", False)),
        stop_grad=bool(target_cfg.get("normalize_reg_stop_grad", True)),
    )

    # β-NLL (Seitzer et al. 2022 — للـ Gaussian؛ تطبيقه على NIG امتداد يحتاج مقارنة عملية):
    # نستخدم w_St² كتباين وليس β/(α-1): الأخير ينفجر عند α→1 (وقيمة α بعد القصّ هي 1 بالضبط)،
    # بينما w_St² منتهٍ لكل α>0 وهو نفسه بديل الـ aleatoric عند Meinert.
    beta_nll = float(target_cfg.get("beta_nll", 0.0))
    nll_term = nig_base
    if beta_nll > 0.0:
        var = tf.stop_gradient(tf.square(nig_w_st(nu, alpha, beta)))
        nll_term = nig_base * tf.pow(var, beta_nll)

    lambda_reg_name = target_cfg.get("lambda_reg_var", "lambda_reg")
    lambda_reg = scheduled_vars[lambda_reg_name] if lambda_reg_name in scheduled_vars else tf.constant(0.0)

    per_sample = nll_term + tf.cast(lambda_reg, tf.float32) * nig_pen

    # Huber مباشر على mu: يعطي المتوسط إشارة لا تمرّ عبر عدم اليقين (لا يستطيع "تفسير" الخطأ بتضخيم β)
    huber_weight = float(target_cfg.get("huber_weight", 0.0))
    if huber_weight > 0.0:
        delta = float(target_cfg.get("huber_delta", 1.0))
        a = tf.abs(error)
        q = tf.minimum(a, delta)
        per_sample = per_sample + huber_weight * (0.5 * tf.square(q) + delta * (a - q))

    total = _wmean(per_sample, w)

    aleatoric = beta / (alpha - 1.0 + 1e-6)
    epistemic = beta / (nu * (alpha - 1.0) + 1e-6)
    total_unc = aleatoric + epistemic
    confidence_est = 1.0 / (1.0 + total_unc)

    stats = {
        "nig_base": _wmean(nig_base, w),
        "nig_pen": _wmean(nig_pen, w),
        "mae": _wmean(tf.abs(error), w),
        "aleatoric": _wmean(aleatoric, w),
        "epistemic": _wmean(epistemic, w),
        "confidence": _wmean(confidence_est, w),
        "calib_loss": tf.constant(0.0),
    }

    if target_cfg.get("use_calibration_loss", False) and "confidence" in keys:
        calib_loss = compute_confidence_calibration_loss(y_true, gamma, outputs[keys["confidence"]], sample_weight=w)
        lambda_calib_name = target_cfg.get("lambda_calib_var", "lambda_calib")
        lambda_calib = scheduled_vars[lambda_calib_name] if lambda_calib_name in scheduled_vars else tf.constant(0.0)
        total = total + calib_loss * tf.cast(lambda_calib, tf.float32)
        stats["calib_loss"] = calib_loss

    return total, stats


register_task_type(
    "evidential", evidential_task_loss,
    ["nig_base", "nig_pen", "mae", "aleatoric", "epistemic", "confidence", "calib_loss"],
    supports_sample_weight=True,
)


# ───────────────────────────── regression بسيطة ─────────────────────────────

def regression_task_loss(y_true, outputs, target_cfg, scheduled_vars, sample_weight=None):
    keys = target_cfg["output_keys"]
    w = None if sample_weight is None else _flat(sample_weight)
    y_true = _flat(y_true)
    y_pred = _flat(outputs[keys["pred"]])
    loss_kind = target_cfg.get("loss_fn", "mse")

    error = y_true - y_pred
    if loss_kind == "mae":
        per = tf.abs(error)
    elif loss_kind == "huber":
        delta = float(target_cfg.get("huber_delta", 1.0))
        a = tf.abs(error)
        q = tf.minimum(a, delta)
        per = 0.5 * tf.square(q) + delta * (a - q)
    else:  # mse (افتراضي)
        per = tf.square(error)

    return _wmean(per, w), {"mae": _wmean(tf.abs(error), w)}


register_task_type("regression", regression_task_loss, ["mae"], supports_sample_weight=True)


# ───────────────────────────── classification ─────────────────────────────

def classification_task_loss(y_true, outputs, target_cfg, scheduled_vars, sample_weight=None):
    keys = target_cfg["output_keys"]
    logits = outputs[keys["logits"]]
    w = None if sample_weight is None else _flat(sample_weight)

    if target_cfg.get("binary", True):
        y_true_f = _flat(y_true)
        logits_f = _flat(logits)
        # [B,1] لنحصل على خسارة لكل عيّنة (على [B] تُختزل Keras المحور الأخير فتعطي رقمًا واحدًا)
        per = tf.keras.losses.binary_crossentropy(y_true_f[:, None], logits_f[:, None])
        pred_label = tf.cast(logits_f >= 0.5, tf.float32)
        acc = _wmean(tf.cast(tf.equal(pred_label, y_true_f), tf.float32), w)
    else:
        y_true_i = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        per = tf.keras.losses.sparse_categorical_crossentropy(y_true_i, logits)
        pred_label = tf.cast(tf.argmax(logits, axis=-1), tf.int32)
        acc = _wmean(tf.cast(tf.equal(pred_label, y_true_i), tf.float32), w)

    return _wmean(per, w), {"accuracy": acc}


register_task_type("classification", classification_task_loss, ["accuracy"], supports_sample_weight=True)


## 4) الموازنة التلقائية بين المهام (Kendall et al., CVPR 2018)

نسخة معمَّمة تعمل بأي عدد وأي أسماء مهام (بدلًا من `log_var_high/low/close`
المكتوبة يدويًا في الكود القديم). الاسم يُشتق تلقائيًا من `trainer.target_names`
(أي من مفاتيح `config['targets']`).

> ⚠️ **شرط رياضي لهذه الموازنة: خسارة المهمة `L` يجب أن تكون موجبة.** الحدّ
> `0.5·exp(-s)·L + 0.5·s` له أدنى نقطة عند `exp(-s) = 1/L` فقط إن كانت `L > 0`.
> إن كانت `L < 0` (وهذا طبيعي لـ NLL توزيع NIG حين يصبح عدم اليقين صغيراً — ظهر
> فعلياً `nig_base ≈ -2` في تدريب حقيقي) فمشتقّة الحدّ بالنسبة لـ `s` موجبة دائماً:
> `s` تنحدر بلا توقف، ووزن المهمة `exp(-s)` يتضاعف كل حقبة، والخسارة الكلية تنهار
> نحو `-∞` (رُصد: `loss` من `+0.9` إلى `-42` خلال 18 حقبة بينما `val_mae` ثابت
> تماماً). ثلاثة آثار صامتة لذلك: (1) `val_loss` يتحسّن كل حقبة بسبب انجراف `s`
> وحده، فـ`BestModelTracker` يعلن «أفضل جديد» كل حقبة ولا يتوقف مبكراً أبداً؛
> (2) تدرّج رؤوس NIG يطغى على الجذع المشترك تحت `clip_norm` العام، فتُحرَم رؤوس
> التصنيف تدريجياً (أوزانها الفعلية 0.15 ← 0.08)؛ (3) الجذع يُدفَع لتضخيم الدليل
> (`nu`/`alpha`) على عيّنات التدريب — إفراط ثقة لا يُعمَّم (`val_nig_base` يسوء
> منذ الحقبة ~4 بينما `nig_base` للتدريب يتحسّن).
>
> لذلك تُستثنى أنواع المهام المذكورة في
> `config['loss']['uncertainty_weighting_exclude_task_types']` (افتراضياً
> `["evidential"]`) من هذه الطبقة وتُجمَع بوزنها الثابت. هذا لا يُفقِد شيئاً:
> NIG تتعلّم مقياس ضجيجها بنفسها عبر `beta`، فوزن Kendall فوقها زائد رياضياً.
> ويُسجَّل أيضاً `raw_loss` (مجموع خسائر المهام بأوزانها الثابتة، بلا حدود Kendall)
> كمقياس مستقرّ لاختيار أفضل نموذج — `val_loss` مع Kendall يتغيّر حتى بلا أي
> تحسّن حقيقي لأن `s` نفسها تتعلّم.

In [ ]:
# @title 4) طبقة الموازنة التلقائية بين المهام (بأسماء ديناميكية)
class UncertaintyWeightedLoss(tf.keras.layers.Layer):
    """L_total = Σ_i (L_i / (2σ²_i) + log(σ_i)) — σ²_i متعلَّمة لكل مهمة"""

    def __init__(self, task_names: List[str], name="uncertainty_weighted_loss"):
        super().__init__(name=name)
        self.task_names = list(task_names)
        self._log_vars = {}
        for t in self.task_names:
            self._log_vars[t] = self.add_weight(
                name=f"log_var_{t}", shape=(), dtype=tf.float32,
                initializer=tf.keras.initializers.Constant(0.0), trainable=True,
            )

    def call(self, losses: Dict[str, tf.Tensor]):
        total = 0.0
        for t in self.task_names:
            precision = tf.exp(-self._log_vars[t])
            total = total + 0.5 * precision * losses[t] + 0.5 * self._log_vars[t]
        return total

    def get_task_variances(self) -> Dict[str, float]:
        return {t: float(tf.exp(v).numpy()) for t, v in self._log_vars.items()}

    def get_effective_weights(self) -> Dict[str, float]:
        variances = self.get_task_variances()
        total_precision = sum(1.0 / v for v in variances.values())
        return {t: (1.0 / variances[t]) / total_precision for t in self.task_names}


## 5) `GenericTrainer` — قلب الإطار

مدرّب واحد يخدم كل الحالات: هدف واحد أو عشرات الأهداف، evidential أو
regression أو classification أو خليط منها، مع موازنة تلقائية أو بدونها، مع
قيود منطقية أو بدونها. **لا يحتوي على أي اسم هدف صريح** — كل شيء يُقرأ من
`self.config` في وقت البناء (`__init__`)، وتُنشأ عليه الـ metrics والمتغيرات
القابلة للجدولة ديناميكيًا.

كل متغيرات هذا الكائن (`self.model`, `self.optimizer` بعد `compile`,
`self.scheduled_vars`, `self.loss_weights`, `self.uncertainty_layer`,
`self.ckpt_epoch`) هي attributes عادية على كائن `tf.keras.Model`، وبالتالي
تُتتبَّع **تلقائيًا** بواسطة `tf.train.Checkpoint` — هذا بالضبط ما يجعل
استئناف التدريب صحيحًا 100% (القسم 7).

> ✅ **تصحيح مقارنةً بالكود القديم**: في `train_step` كانت التدرّجات تُحسب فقط
> على `self.model.trainable_variables`، فتتجاهل متغيرات طبقة الموازنة التلقائية
> (`log_var` لكل مهمة) — أي أن "الموازنة المتعلَّمة" (Kendall) لم تكن تتحدّث
> بالفعل عبر التدريب! هنا نستخدم `self.trainable_variables` التي تضمّ النموذج
> وطبقة الموازنة معًا.

In [ ]:
# @title 5) المدرّب العام (GenericTrainer)
class GenericTrainer(tf.keras.Model):
    def __init__(self, model: tf.keras.Model, config: dict, **kwargs):
        super().__init__(**kwargs)
        self.model = model
        self.config = config
        self.target_names = list(config["targets"].keys())

        # ── معاملات قابلة للجدولة (بأي اسم عرّفه المستخدم في config['loss']['schedules']) ──
        self.scheduled_vars: Dict[str, tf.Variable] = {}
        for pname, pcfg in config["loss"].get("schedules", {}).items():
            self.scheduled_vars[pname] = tf.Variable(
                float(pcfg.get("start", 0.0)), trainable=False, dtype=tf.float32, name=pname
            )

        # ── وزن كل هدف (قابل للتحديث الديناميكي بواسطة TaskWeightUpdater) ──
        self.loss_weights: Dict[str, tf.Variable] = {
            t: tf.Variable(float(tcfg.get("loss_weight", 1.0)), trainable=False,
                            dtype=tf.float32, name=f"loss_weight_{t}")
            for t, tcfg in config["targets"].items()
        }

        # ── عدد الحقب المنجزة — جزء من حالة الـ checkpoint نفسها (وليس فقط ملف JSON خارجي) ──
        # يُحدَّث في نهاية كل حقبة (EpochCheckpointCallback) وهو المرجع الوحيد لـ initial_epoch.
        self.ckpt_epoch = tf.Variable(0, trainable=False, dtype=tf.int64, name="ckpt_epoch")

        # ── موازنة تلقائية بين المهام (اختيارية) ──
        # تُطبَّق فقط على المهام ذات الخسارة الموجبة؛ المستثناة (evidential افتراضياً) تُجمَع بوزنها الثابت
        # (راجع ملاحظة القسم 4: Kendall مع خسارة سالبة تنهار نحو -∞).
        self.use_uncertainty_weighting = bool(config["loss"].get("use_uncertainty_weighting", False))
        excluded_types = set(config["loss"].get("uncertainty_weighting_exclude_task_types", ["evidential"]))
        self.kendall_task_names = [
            t for t, tcfg in config["targets"].items()
            if self.use_uncertainty_weighting and tcfg["task_type"] not in excluded_types
        ]
        self.fixed_task_names = [t for t in self.target_names if t not in self.kendall_task_names]
        self.uncertainty_layer = (
            UncertaintyWeightedLoss(self.kendall_task_names) if self.kendall_task_names else None
        )

        # ── قيود منطقية/فيزيائية اختيارية بين المخرجات ──
        self.constraints_cfg = config["loss"].get("constraints", [])

        self._create_metrics()

    # ─────────────────────────────────────────────────────────────
    def _create_metrics(self):
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        # مجموع خسائر المهام بأوزانها الثابتة (بلا حدود Kendall) — مقياس مستقرّ لاختيار أفضل نموذج
        self.raw_loss_tracker = tf.keras.metrics.Mean(name="raw_loss")
        self._target_loss_trackers = {t: tf.keras.metrics.Mean(name=f"loss_{t}") for t in self.target_names}

        self._stat_trackers: Dict[str, tf.keras.metrics.Mean] = {}
        for t, tcfg in self.config["targets"].items():
            task_type = tcfg["task_type"]
            for key in TASK_REGISTRY[task_type]["stat_keys"]:
                self._stat_trackers[f"{t}_{key}"] = tf.keras.metrics.Mean(name=f"{t}_{key}")

        self._constraint_trackers = {
            c["name"]: tf.keras.metrics.Mean(name=f"constraint_{c['name']}")
            for c in self.constraints_cfg
        }

    @property
    def metrics(self):
        return (
            [self.loss_tracker, self.raw_loss_tracker]
            + list(self._target_loss_trackers.values())
            + list(self._stat_trackers.values())
            + list(self._constraint_trackers.values())
        )

    # ─────────────────────────────────────────────────────────────
    @staticmethod
    def _weight_for_target(sample_weight, target_name: str, tcfg: dict):
        """sample_weight إما موتّر واحد (يسري على كل الأهداف) أو dict بمفتاح اسم الهدف أو true_key."""
        if sample_weight is None:
            return None
        if isinstance(sample_weight, collections.abc.Mapping):
            for key in (target_name, tcfg["true_key"]):
                if key in sample_weight:
                    return sample_weight[key]
            return None
        return sample_weight

    def _compute_losses(self, x, y, sample_weight, training):
        outputs = self.model(x, training=training)

        per_target_losses = {}
        per_stat_values: Dict[str, tf.Tensor] = {}

        for t, tcfg in self.config["targets"].items():
            y_true_t = y[tcfg["true_key"]]
            entry = TASK_REGISTRY[tcfg["task_type"]]
            w_t = self._weight_for_target(sample_weight, t, tcfg)
            if w_t is not None:
                if not entry["supports_sample_weight"]:
                    raise ValueError(
                        f"❌ الهدف '{t}' من النوع '{tcfg['task_type']}' لا يدعم sample_weight "
                        f"(سجّله بـ supports_sample_weight=True وأضف وسيط sample_weight لدالته) — "
                        f"رفضنا تجاهله بصمت لأن هذا يُفسد التدريب دون أي إنذار."
                    )
                raw_loss, stats = entry["fn"](y_true_t, outputs, tcfg, self.scheduled_vars, sample_weight=w_t)
            else:
                raw_loss, stats = entry["fn"](y_true_t, outputs, tcfg, self.scheduled_vars)

            w = self.loss_weights[t]
            per_target_losses[t] = tf.cast(raw_loss, tf.float32) * w
            for k, v in stats.items():
                per_stat_values[f"{t}_{k}"] = tf.cast(v, tf.float32)

        # القيود المنطقية لا تُرجَّح بـ sample_weight (دالتها تأخذ outputs فقط)
        per_constraint_values = {}
        constraint_total = tf.constant(0.0, tf.float32)
        for c in self.constraints_cfg:
            penalty = tf.cast(c["fn"](outputs), tf.float32)
            weight = self.scheduled_vars.get(c.get("weight_var"), tf.constant(1.0))
            constraint_total = constraint_total + penalty * tf.cast(weight, tf.float32)
            per_constraint_values[c["name"]] = penalty

        raw_loss = tf.add_n(list(per_target_losses.values()))
        if self.uncertainty_layer is not None:
            total_loss = self.uncertainty_layer({t: per_target_losses[t] for t in self.kendall_task_names})
            if self.fixed_task_names:
                total_loss = total_loss + tf.add_n([per_target_losses[t] for t in self.fixed_task_names])
        else:
            total_loss = raw_loss

        extra = constraint_total
        if self.model.losses:
            extra = extra + tf.add_n([tf.cast(l, tf.float32) for l in self.model.losses])
        total_loss = total_loss + extra
        raw_loss = raw_loss + extra

        return total_loss, raw_loss, per_target_losses, per_stat_values, per_constraint_values

    # ─────────────────────────────────────────────────────────────
    def _update_trackers(self, total_loss, raw_loss, per_target_losses, per_stat_values, per_constraint_values):
        self.loss_tracker.update_state(total_loss)
        self.raw_loss_tracker.update_state(raw_loss)
        for t, v in per_target_losses.items():
            self._target_loss_trackers[t].update_state(v)
        for k, v in per_stat_values.items():
            if k in self._stat_trackers:
                self._stat_trackers[k].update_state(v)
        for name, v in per_constraint_values.items():
            self._constraint_trackers[name].update_state(v)

    def train_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        # self.trainable_variables تشمل النموذج + متغيرات طبقة الموازنة (log_var لكل مهمة)
        train_vars = self.trainable_variables
        with tf.GradientTape() as tape:
            total_loss, raw_loss, per_target, stats, constraints = self._compute_losses(x, y, sample_weight, training=True)

        grads = tape.gradient(total_loss, train_vars)
        # القص (global_clipnorm) يتم داخل الـ optimizer نفسه — لا قص يدوي مكرر هنا.
        self.optimizer.apply_gradients(zip(grads, train_vars))

        self._update_trackers(total_loss, raw_loss, per_target, stats, constraints)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        total_loss, raw_loss, per_target, stats, constraints = self._compute_losses(x, y, sample_weight, training=False)
        self._update_trackers(total_loss, raw_loss, per_target, stats, constraints)
        return {m.name: m.result() for m in self.metrics}


## 6) الكولباكس العامة (Callbacks)

كل كولباك هنا **عامة** بالمعنى الحرفي: لا تعرف شيئًا عن أسماء أهدافك.

- `ParamScheduler`: كولباك **واحدة تكفي كل المعاملات القابلة للجدولة** —
  تُغني عن كتابة كلاس منفصل لكل معامل (`lambda_reg`, `lambda_calib`,
  `penalty_weight`, ...). الجدولة دالة محضة في رقم الحقبة → **آمنة تمامًا عند
  الاستئناف**، لأن Keras يستدعي `on_epoch_begin` بنفس رقم الحقبة الصحيح حتى
  بعد `resume` (لا حاجة لحفظ حالتها في الـ checkpoint إطلاقًا).
- جدولة معدل التعلّم (`build_lr_schedule_fn`) لنفس السبب: دالة محضة في الحقبة.
- `BestModelTracker`: مصدر واحد لتعريف «الأفضل» + التوقف المبكر معًا (يحلّ محل
  `SmoothedEarlyStopping` و`BestWeightsSaver` القديمتين اللتين كانتا تحسبان «الأفضل» بطريقتين
  مختلفتين ويفقدان حالتهما عند الاستئناف). **قابلة للاستئناف الكامل** عبر `get_state()`/`load_state()`
  + ملف `best_meta.json` مستقل يُقرأ عند البناء.
- `TaskWeightUpdater`: تحديث وزن كل هدف تلقائيًا نسبةً لصعوبته الحالية — بديل
  يدوي اختياري (فقط إذا عطّلت الموازنة التلقائية Kendall).
- `MetricsLogger`: ملخص دوري + `history` قابل للاستئناف.
- `SnapshotEnsemble` / `BestWeightsSaver`: حفظ لقطات دورية / أفضل أوزان
  للنشر النهائي (اختياريتان تمامًا).

In [ ]:
# @title 6.1) ParamScheduler + جدولة معدل التعلّم
def _progress_curve(p: float, schedule: str) -> float:
    if schedule == "cosine":
        return 0.5 * (1 - np.cos(np.pi * p))
    elif schedule == "exponential":
        return p ** 2
    elif schedule == "sqrt":
        return np.sqrt(p)
    return p  # linear


def schedule_value(start: float, end: float, warmup_epochs: int, schedule: str, epoch: int) -> float:
    """قيمة معامل مجدول عند حقبة معيّنة — دالة محضة في رقم الحقبة (هذا ما يجعل الاستئناف آمنًا)."""
    warmup_epochs = max(int(warmup_epochs), 0)
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return end
    return start + (end - start) * _progress_curve(epoch / warmup_epochs, schedule)


class ParamScheduler(tf.keras.callbacks.Callback):
    """جدولة عامة لأي tf.Variable — تُستخدم لأي معامل مجدول عرّفته في config['loss']['schedules']"""

    def __init__(self, var: tf.Variable, start: float, end: float, warmup_epochs: int,
                 schedule: str = "linear", label: Optional[str] = None, log_every: int = 5, verbose: int = 1):
        super().__init__()
        self.var = var
        self.start = start
        self.end = end
        self.warmup_epochs = max(int(warmup_epochs), 0)
        self.schedule = schedule
        self.label = label or var.name
        self.log_every = log_every
        self.verbose = verbose

    def value_at(self, epoch: int) -> float:
        return schedule_value(self.start, self.end, self.warmup_epochs, self.schedule, epoch)

    def on_epoch_begin(self, epoch, logs=None):
        new_value = self.value_at(epoch)
        self.var.assign(new_value)
        if self.verbose and (epoch % self.log_every == 0):
            print(f"🔄 [{self.label}] Epoch {epoch + 1}: {new_value:.6f}")


def apply_schedules(trainer, config: dict, epoch: int):
    """يضبط كل المعاملات المجدولة فورًا على قيمتها عند الحقبة `epoch` (قبل أول fit) —
    كي يرى أي evaluate() أو استدلال قبل التدريب القيم الصحيحة، لا قيم البداية."""
    for pname, pcfg in config["loss"].get("schedules", {}).items():
        trainer.scheduled_vars[pname].assign(schedule_value(
            pcfg["start"], pcfg["end"], pcfg.get("warmup_epochs", 0), pcfg.get("schedule", "linear"), epoch))


def cosine_warm_restarts(epoch, lr_initial, lr_min, cycle_length=10, cycle_mult=1.5):
    cycle_epoch = epoch
    current_len = cycle_length
    while cycle_epoch >= current_len:
        cycle_epoch -= current_len
        current_len = int(current_len * cycle_mult)
    progress = cycle_epoch / current_len
    return lr_min + 0.5 * (lr_initial - lr_min) * (1 + np.cos(np.pi * progress))


def build_lr_schedule_fn(opt_cfg: dict, rewarm_from_epoch: Optional[int] = None,
                         rewarm_epochs: int = 0) -> Callable[[int], float]:
    """يبني دالة lr(epoch) بالكامل من config['optimizer'] — دالة محضة، آمنة عند الاستئناف.
    rewarm_*: للبدء الدافئ (warm_start) فقط: optimizer جديد ⇒ عزوم Adam صفرية، فنرفع lr تدريجيًا
    خلال rewarm_epochs حقبة بدل القفز مباشرة إلى القيمة الكاملة فوق أوزان متقاربة."""
    lr_initial = opt_cfg["lr_initial"]
    lr_min = opt_cfg["lr_min"]
    warmup = int(opt_cfg.get("lr_warmup_epochs", 0))
    sched = opt_cfg.get("lr_schedule", {"type": "constant"})

    def base(epoch):
        if warmup and epoch < warmup:
            return lr_initial * (epoch + 1) / warmup
        e = epoch - warmup
        if sched["type"] == "cosine_restarts":
            return cosine_warm_restarts(e, lr_initial, lr_min, sched.get("cycle_length", 10), sched.get("cycle_mult", 1.5))
        elif sched["type"] == "cosine":
            total = sched.get("total_epochs", 100)
            progress = min(e / max(total, 1), 1.0)
            return lr_min + 0.5 * (lr_initial - lr_min) * (1 + np.cos(np.pi * progress))
        else:  # constant
            return lr_initial

    def fn(epoch):
        lr = base(epoch)
        if rewarm_from_epoch is not None and rewarm_epochs > 0:
            k = epoch - rewarm_from_epoch + 1          # 1 = أول حقبة بعد البدء الدافئ
            if 1 <= k <= rewarm_epochs:
                lr = lr * k / (rewarm_epochs + 1)
        return lr

    return fn


In [ ]:
# @title 6.2) BestModelTracker — «أفضل نسخة» + التوقف المبكر في مكان واحد (قابل للاستئناف الكامل)
class BestModelTracker(tf.keras.callbacks.Callback):
    """يمتلك وحده تعريف «الأفضل» — بديل SmoothedEarlyStopping + BestWeightsSaver اللذين كانا يحسبانه بطريقتين
    مختلفتين (الأول بمتوسط نافذة، الثاني بالقيمة الخام) وكلاهما يفقد حالته عند الاستئناف.

    smoothing: كيف يُنعَّم المقياس قبل المقارنة —
        "none"   : القيمة الخام في كل حقبة
        "window" : متوسط آخر `window` حقب (السلوك القديم)
        "ema"    : متوسط أسّي متحرك  s_t = beta*s_(t-1) + (1-beta)*x_t   (s_1 = x_1)
    الحفظ: أفضل أوزان النموذج الأساسي تُكتب ذرّيًا إلى best_path مع best_meta.json (قيمة الأفضل وحقبته).
    الاستئناف: قيمة الأفضل تُقرأ من best_meta.json — فلا تُكتب فوق الملف حقبةٌ أسوأ بعد انقطاع Colab.
    الاسترجاع: يتم في on_train_end (بعد أن يبدّل Keras أوزان EMA الخاصة بالـ optimizer داخل fit)، من الملف على القرص
    فيعمل حتى لو لم يتحسّن شيء في الجلسة الحالية."""

    def __init__(self, base_model: tf.keras.Model, best_path: Optional[str] = None,
                 monitor: str = "val_loss", mode: str = "min", patience: int = 15, min_delta: float = 1e-4,
                 smoothing: str = "window", window: int = 3, ema_beta: float = 0.7,
                 restore_best_weights: bool = True, verbose: int = 1, initial_state: Optional[dict] = None,
                 trainer: Optional["GenericTrainer"] = None, weights_snapshot: str = "raw"):
        super().__init__()
        if smoothing not in ("none", "window", "ema"):
            raise ValueError("smoothing يجب أن يكون none|window|ema")
        if mode not in ("min", "max"):
            raise ValueError("mode يجب أن يكون min|max")
        if weights_snapshot not in ("raw", "ema_weights"):
            raise ValueError("weights_snapshot يجب أن يكون raw|ema_weights")
        if weights_snapshot == "ema_weights" and trainer is None:
            raise ValueError("weights_snapshot='ema_weights' يتطلب تمرير trainer=")
        self.trainer = trainer
        self.weights_snapshot = weights_snapshot
        self.base_model = base_model
        self.best_path = best_path
        self.best_meta_path = os.path.join(os.path.dirname(best_path), "best_meta.json") if best_path else None
        self.monitor, self.mode = monitor, mode
        self.patience, self.min_delta = int(patience), float(min_delta)
        self.smoothing, self.window, self.ema_beta = smoothing, max(int(window), 1), float(ema_beta)
        self.restore_best_weights = restore_best_weights
        self.verbose = verbose

        self.history: List[float] = []
        self.ema: Optional[float] = None
        self.best: Optional[float] = None
        self.best_epoch: Optional[int] = None
        self.wait = 0
        self._mem_weights = None
        if best_path:
            os.makedirs(os.path.dirname(best_path), exist_ok=True)

        if initial_state:
            self.load_state(initial_state)
        self._load_best_meta()

    # ── الحالة (تُحفظ في meta.json عبر EpochCheckpointCallback) ─────────────────
    def get_state(self) -> dict:
        return {"monitor": self.monitor, "mode": self.mode, "smoothing": self.smoothing,
                "history": self.history[-200:], "ema": self.ema,
                "best": self.best, "best_epoch": self.best_epoch, "wait": int(self.wait)}

    def load_state(self, state: dict):
        if state.get("monitor") not in (None, self.monitor) or state.get("mode") not in (None, self.mode):
            self._metric_definition_changed(f"{state.get('monitor')}/{state.get('mode')}")
            return
        self.history = list(state.get("history", []))
        self.ema = state.get("ema")
        b = state.get("best")
        self.best = float(b) if b is not None and np.isfinite(b) else None   # الحالة القديمة كانت تحفظ inf
        self.best_epoch = state.get("best_epoch")
        self.wait = int(state.get("wait", 0))
        print(f"🔄 [BestModelTracker] استُرجعت الحالة: best={self.best} (حقبة {self.best_epoch}) wait={self.wait}")

    def _load_best_meta(self):
        """best_meta.json يُكتب مع ملف الأوزان نفسه، فهو المصدر الأصدق لقيمة الأفضل."""
        if not (self.best_meta_path and os.path.exists(self.best_meta_path) and os.path.exists(self.best_path)):
            return
        try:
            with open(self.best_meta_path, encoding="utf-8") as f:
                meta = json.load(f)
        except Exception as e:
            print(f"⚠️ [BestModelTracker] تعذّر قراءة best_meta.json ({e}) — سيُعاد تقييم الأفضل")
            return
        if meta.get("monitor") != self.monitor or meta.get("mode") != self.mode:
            self._metric_definition_changed(f"{meta.get('monitor')}/{meta.get('mode')}")
            return
        self.best, self.best_epoch = float(meta["best"]), meta.get("best_epoch")

    def _metric_definition_changed(self, old: str):
        """قيم مقياس مختلف لا تُقارن بقيم مقياس آخر: نبدأ التتبّع من جديد ونحتفظ بنسخة من الملف القديم."""
        print(f"⚠️ [BestModelTracker] تغيّر مقياس الاختيار ({old} → {self.monitor}/{self.mode}) — يبدأ التتبّع من جديد")
        if self.best_path and os.path.exists(self.best_path):
            backup = self.best_path.replace(".weights.h5", ".prev.weights.h5")
            os.replace(self.best_path, backup)
            print(f"   نسخة الأفضل القديمة محفوظة في: {backup}")
        for p in (self.best_meta_path,):
            if p and os.path.exists(p):
                os.remove(p)
        self.history, self.ema, self.best, self.best_epoch, self.wait = [], None, None, None, 0

    # ── المنطق ─────────────────────────────────────────────────────────────────
    def _smooth(self, current: float) -> float:
        self.history.append(current)
        if self.smoothing == "ema":
            self.ema = current if self.ema is None else self.ema_beta * self.ema + (1.0 - self.ema_beta) * current
            return float(self.ema)
        if self.smoothing == "window":
            return float(np.mean(self.history[-self.window:]))
        return float(current)

    def _is_better(self, s: float) -> bool:
        if self.best is None:
            return True
        return s < self.best - self.min_delta if self.mode == "min" else s > self.best + self.min_delta

    def _snapshot_weights(self) -> Optional[list]:
        """يُرجع أوزان EMA (المتوسط المتحرك للـ optimizer) إن طُلب ذلك وكان متاحًا، وإلا None (يعني: استخدم الأوزان الخام كما هي)."""
        if self.weights_snapshot != "ema_weights":
            return None
        opt = self.trainer.optimizer
        if not getattr(opt, "use_ema", False):
            print("⚠️ [BestModel] weights_snapshot='ema_weights' لكن optimizer.use_ema=False — استُخدمت الأوزان الخام بدلًا")
            return None
        shadow = getattr(opt, "_model_variables_moving_average", None)
        if not shadow:
            return None
        # مطابقة بالهوية (id) بين trainer.trainable_variables (ما بُني عليه EMA) وأوزان base_model —
        # نستخدم id() لا الاسم لأن Keras قد يكرر أسماء الطبقات؛ للأوزان غير القابلة للتدريب (BatchNorm) نُبقي القيمة الخام.
        mapping = {id(v): av for v, av in zip(self.trainer.trainable_variables, shadow)}
        return [(mapping[id(w)].numpy() if id(w) in mapping else w.numpy()) for w in self.base_model.weights]

    def _save_best(self):
        ema_values = self._snapshot_weights()
        if ema_values is not None:
            raw_values = self.base_model.get_weights()
            self.base_model.set_weights(ema_values)
        try:
            if not self.best_path:
                self._mem_weights = self.base_model.get_weights()
                return
            tmp = os.path.join(os.path.dirname(self.best_path), "best.tmp.weights.h5")
            self.base_model.save_weights(tmp)
            os.replace(tmp, self.best_path)
            atomic_write_json(self.best_meta_path, {
                "best": self.best, "best_epoch": self.best_epoch, "monitor": self.monitor, "mode": self.mode,
                "smoothing": self.smoothing, "weights_snapshot": self.weights_snapshot,
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")})
        finally:
            if ema_values is not None:
                self.base_model.set_weights(raw_values)   # لا نترك تدريب الحقبة القادمة يبدأ من أوزان EMA بالخطأ

    def on_epoch_end(self, epoch, logs=None):
        current = (logs or {}).get(self.monitor)
        if current is None or not np.isfinite(current):
            return
        smoothed = self._smooth(float(current))

        if self._is_better(smoothed):
            self.best, self.best_epoch, self.wait = smoothed, epoch + 1, 0
            self._save_best()
            if self.verbose:
                print(f"   ⭐ [BestModel] أفضل جديد: {self.monitor}={current:.5f} (smoothed={smoothed:.5f}) — حقبة {epoch + 1}")
        else:
            self.wait += 1

        if self.verbose:
            print(f"   [BestModel] {self.monitor}={current:.5f} (smoothed={smoothed:.5f}) "
                  f"best={self.best:.5f}@{self.best_epoch} wait={self.wait}/{self.patience}")

        if self.wait >= self.patience:
            self.model.stop_training = True
            print(f"⏹️ Early stopping عند الحقبة {epoch + 1}")

    def restore_best(self) -> bool:
        """يعيد أفضل أوزان إلى النموذج الأساسي (من القرص إن وُجد، وإلا من الذاكرة). يُرجع True عند النجاح."""
        if self.best_path and os.path.exists(self.best_path):
            self.base_model.load_weights(self.best_path)
            return True
        if self._mem_weights is not None:
            self.base_model.set_weights(self._mem_weights)
            return True
        return False

    def on_train_end(self, logs=None):
        # يعمل عند أي نهاية تدريب (توقف مبكر أو اكتمال الحقب)، وبعد تبديل EMA الذي يجريه Keras داخل fit()
        if self.restore_best_weights and self.best is not None:
            if self.restore_best():
                print(f"✅ [BestModel] استُرجعت أفضل أوزان (حقبة {self.best_epoch}, {self.monitor} المنعَّم={self.best:.5f})")
            else:
                print("⚠️ [BestModel] لا توجد نسخة أوزان مخزَّنة للأفضل — بقيت الأوزان الحالية")


In [ ]:
# @title 6.3) TaskWeightUpdater + MetricsLogger
class TaskWeightUpdater(tf.keras.callbacks.Callback):
    """تحديث وزن كل هدف تلقائيًا نسبةً لصعوبته الحالية — يعمل مع أي عدد/أسماء أهداف.
    استخدمها فقط إذا عطّلت use_uncertainty_weighting (الطريقتان تتعارضان منطقيًا)."""

    def __init__(self, trainer: "GenericTrainer", update_frequency=1, verbose=1):
        super().__init__()
        self.trainer = trainer
        self.target_names = trainer.target_names
        self.update_frequency = max(int(update_frequency), 0)
        self.verbose = verbose

    def on_epoch_end(self, epoch, logs=None):
        if self.update_frequency == 0 or (epoch + 1) % self.update_frequency != 0:
            return
        logs = logs or {}
        losses = {t: float(logs.get(f"loss_{t}", 0.0)) for t in self.target_names}
        total = sum(losses.values())
        if total <= 0:
            return
        n = len(self.target_names)
        for t, l in losses.items():
            self.trainer.loss_weights[t].assign(n * l / total)
        if self.verbose:
            s = ", ".join(f"{t}={self.trainer.loss_weights[t].numpy():.2f}" for t in self.target_names)
            print(f"📊 [TaskWeights] {s}")


class MetricsLogger(tf.keras.callbacks.Callback):
    """ملخص تدريب دوري + history قابل للاستئناف عبر get_state/load_state.

    class_baselines: {مفتاح هدف تصنيف: نسبة الفئة الأغلب في val} اختياري —
    عام تماماً (المفتاح أي اسم هدف يظهر بمفاتيح `{مفتاح}_accuracy`/
    `val_{مفتاح}_accuracy` في logs، لا شيء خاص بمشروع بعينه). إن مُرِّر، يُطبَع
    مقابل val_accuracy المقابلة في كل ملخّص — نفس مبدأ `tree_naive_baseline_accuracy`
    في `detect_success_failure_patterns` (دقّة خام قد تبدو جيدة وهي فعلياً لا
    تتجاوز تخمين الفئة الأغلب بلا أي مهارة حقيقية، خصوصاً مع أهداف غير متوازنة)."""

    def __init__(self, log_every=5, verbose=1, initial_state=None, class_baselines=None):
        super().__init__()
        self.log_every = log_every
        self.verbose = verbose
        self.class_baselines = class_baselines or {}
        self.history: Dict[str, list] = {"epoch": [], "loss": [], "val_loss": []}
        if initial_state:
            self.load_state(initial_state)

    def get_state(self) -> dict:
        return self.history

    def load_state(self, state: dict):
        if state:
            self.history = state

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.history["epoch"].append(epoch + 1)
        self.history["loss"].append(float(logs.get("loss", 0)))
        self.history["val_loss"].append(float(logs.get("val_loss", 0)))

        if not self.verbose or (epoch + 1) % self.log_every != 0:
            return

        print(f"\n{'=' * 70}\n📊 ملخص Epoch {epoch + 1}\n{'=' * 70}")
        for k, v in logs.items():
            try:
                print(f"   {k:<22}: {float(v):.5f}")
            except (TypeError, ValueError):
                print(f"   {k:<22}: {v}")

        if self.class_baselines:
            print("   — مقارنة بخطّ أساس الفئة الأغلب (val، لا مهارة حقيقية إن لم يُتجاوَز) —")
            for t, baseline in self.class_baselines.items():
                val_acc = logs.get(f"val_{t}_accuracy")
                if val_acc is None:
                    continue
                beats = float(val_acc) > baseline + 0.01
                verdict = "✅ فوق خطّ الأساس" if beats else "⚠️ عند/تحت خطّ الأساس"
                print(f"   {t:<22}: val_accuracy={float(val_acc):.3f} مقابل خطّ أساس={baseline:.3f}  {verdict}")

        trainer = self.model  # داخل fit()، self.model هو الـ GenericTrainer نفسه
        if getattr(trainer, "uncertainty_layer", None) is not None:
            weights = trainer.uncertainty_layer.get_effective_weights()
            print("   أوزان المهام الفعلية : " + ", ".join(f"{t}={w:.2f}" for t, w in weights.items()))
            if trainer.fixed_task_names:
                print("   مهام بوزن ثابت (خارج Kendall): " + ", ".join(trainer.fixed_task_names))
        if getattr(trainer, "scheduled_vars", None):
            sched_str = ", ".join(f"{k}={v.numpy():.5f}" for k, v in trainer.scheduled_vars.items())
            print(f"   المعاملات المجدولة    : {sched_str}")
        print(f"{'=' * 70}\n")


def _test_metrics_logger_class_baselines():
    """يتحقّق من إضافة class_baselines إلى MetricsLogger (24 سبتمبر 2026):
    (أ) بلا class_baselines، السلوك القديم تماماً — لا سطر مقارنة يُطبَع،
    (ب) بها، يُطبَع سطر مقارنة صريح لكل هدف تصنيف موجود في logs، يُصنِّف
    val_accuracy كـ"فوق"/"عند أو تحت" خطّ الأساس بمقارنة رقمية مباشرة، لا
    نصّاً ثابتاً. يلتقط stdout بدل تشغيل تدريب حقيقي (لا حاجة لبيانات)."""
    import io, contextlib

    logs = {
        'loss': 1.0, 'val_loss': -1.0,
        'close_class_accuracy': 0.77, 'val_close_class_accuracy': 0.498,
        'high_class_accuracy': 0.80, 'val_high_class_accuracy': 0.75,
    }

    logger_no_baseline = MetricsLogger(log_every=1, verbose=1)
    logger_no_baseline.set_model(None)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        logger_no_baseline.on_epoch_end(0, dict(logs))
    assert 'خطّ الأساس' not in buf.getvalue(), (
        'بلا class_baselines يجب ألا يظهر أي سطر مقارنة — تغيير سلوك قديم غير مقصود.')

    logger_with_baseline = MetricsLogger(
        log_every=1, verbose=1,
        class_baselines={'close_class': 0.50, 'high_class': 0.60})
    logger_with_baseline.set_model(None)
    buf2 = io.StringIO()
    with contextlib.redirect_stdout(buf2):
        logger_with_baseline.on_epoch_end(0, dict(logs))
    out = buf2.getvalue()
    assert 'val_accuracy=0.498' in out and '⚠️' in out, (
        'close_class (val=0.498 مقابل خطّ أساس 0.50) يجب أن يُصنَّف "عند/تحت خطّ الأساس".')
    assert 'val_accuracy=0.750' in out and '✅' in out, (
        'high_class (val=0.75 مقابل خطّ أساس 0.60) يجب أن يُصنَّف "فوق خطّ الأساس".')
    print("✅ _test_metrics_logger_class_baselines: التوافق الخلفي محفوظ، والمقارنة صحيحة عددياً.")
    return True


_test_metrics_logger_class_baselines()

In [ ]:
# @title 6.4) SnapshotEnsemble (اختياري)
class SnapshotEnsemble(tf.keras.callbacks.Callback):
    """حفظ لقطات (snapshots) في حقب محددة — لبناء ensemble لاحقًا.
    (BestWeightsSaver حُذفت: وظيفتها صارت جزءًا من BestModelTracker في 6.2)"""

    def __init__(self, save_epochs: List[int], save_dir: str, base_model: tf.keras.Model, verbose=1):
        super().__init__()
        self.save_epochs = set(save_epochs)
        self.save_dir = save_dir
        self.base_model = base_model
        self.verbose = verbose
        if self.save_epochs:
            os.makedirs(save_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) in self.save_epochs:
            path = os.path.join(self.save_dir, f"snapshot_epoch_{epoch + 1}.weights.h5")
            self.base_model.save_weights(path)
            if self.verbose:
                print(f"💾 [Snapshot] {path}")


## 7) نظام الحفظ والاستئناف — حلّ المشكلة المطلوبة تحديدًا

### جوهر المشكلة القديمة
كان يتم إنشاء **مدرّب جديد** (`AdvancedTrainer`) عند الاستئناف، ثم تحميل أوزان
النموذج فقط (`base_model.load_weights(...)`)، بينما `optimizer` (Adam/AdamW)
يُبنى من جديد بحالة ابتدائية صفرية (momentum = 0, variance = 0).

### الحل
1. **نحفظ كل شيء معًا**: `tf.train.Checkpoint(trainer=trainer)` يتتبّع تلقائيًا: أوزان `self.model`،
   `self.optimizer` بالكامل (بما فيها slots الزخم/الفارينس وEMA)، كل `scheduled_vars`/`loss_weights`/
   `uncertainty_layer`، و`self.ckpt_epoch`.

2. **بناء المتغيرات قبل الاسترجاع — بلا أي تدريب فعلي (🆕 مُصلَح)**: الاسترجاع يحتاج أن تكون كل
   المتغيرات (بما فيها *slots* الـ optimizer) موجودة مسبقًا. النسخة القديمة كانت تُنشئها بتنفيذ
   `train_on_batch` تجريبي ثم تتراجع يدويًا عن الأوزان القابلة للتدريب — لكن التراجع لا يشمل كل شيء:
   `optimizer.iterations` يبقى 1 لا 0، عزوم Adam **الداخلية** (لا الأوزان) لا تُعاد، وطبقات مثل
   `BatchNormalization` تُحدِّث إحصاءاتها المتحركة بلا أي تراجع عنها إطلاقًا. الآن `build_trainer_variables`
   تستدعي تمريرًا أماميًا واحدًا بـ`training=False` (لا يُحدِّث BatchNorm) ثم `optimizer.build(...)` فقط
   (يُنشئ الـ slots دون أي `apply_gradients`) — فلا يوجد أي أثر جانبي يحتاج تراجعًا عنه أصلًا.

3. **منع إعادة استخدام مدرّب بحالة غير متطابقة مع config الحالي (🆕 مُصلَح)**: `TRAINER_REGISTRY` كان
   يُعيد أي مدرّب مخزَّن لنفس `run_dir` بصرف النظر عن `config` الممرَّر — فتغيير `config` (مثلًا تفعيل
   الموازنة التلقائية) في خلية ثم إعادة تشغيلها في نفس الجلسة لا يُطبَّق. الآن تُقارَن بصمة `config`
   (`config_fingerprint`) قبل إعادة الاستخدام؛ أي اختلاف يُعيد البناء تلقائيًا.

4. **`config['run']['train_mode']` (🆕)**: بدل التخمين الضمني (checkpoint موجود ⇒ استئناف)، يمكنك
   التحكم صراحة: `auto` (نفس السلوك القديم) | `new` (يبدأ من الصفر، وأي حالة سابقة **تُؤرشَف لا تُحذف**
   في `run_dir/_archive/<timestamp>/`) | `resume` (يفشل بوضوح إن لم توجد حالة، بدل بدء صامت من الصفر) |
   `warm_start` (انظر البند التالي).

5. **`warm_start` — حلّ مباشر لمشكلة «عقوبات الثقة تبدأ من صفر»**: عندما تكمل التدريب من نموذج مُنجز
   سابقًا بطريقة `load_weights` بسيطة (لا عبر checkpoint هذا الإطار)، فإن `config['run']['train_mode']
   = 'warm_start'` مع `config['run']['warm_start'] = {'weights_path': ..., 'epochs_done': N}` يضبط
   **فورًا** كل معاملات `config['loss']['schedules']` (مثل `lambda_reg`, `lambda_calib`) على قيمتها عند
   الحقبة N بدل الصفر — وهذا يمنع بالضبط انهيار الموثوقية الذي وصفتَه (النموذج يُدرَّب حقبة إضافية
   بلا أي عقوبة ثقة فجأة). `optimizer` يبقى جديدًا بالضرورة (لا توجد حالته المحفوظة)، لذا `lr_rewarmup_epochs`
   اختياري لتسخين قصير لمعدل التعلّم بدل قفزه لقيمته الكاملة فوق عزوم Adam صفرية.

### ⚠️ تنبيه حاسم بخصوص "التخزين المؤقت" لـ Colab
مسار `/content/...` (بما فيه `/content/drive` **إن لم يُركَّب Drive فعليًا**) يُمسح بالكامل عند
انقطاع/انتهاء الجلسة. **لضمان الاستئناف عبر جلسات مختلفة، يجب أن يكون `run_dir` مسارًا دائمًا** — الأبسط:
مسار داخل `/content/drive/MyDrive/...` (بعد تركيب Drive).

In [ ]:
# @title 7.1) CheckpointManager + أرشفة الحالة + بناء المتغيرات دون تحديث
TRAINER_REGISTRY: Dict[str, Dict[str, Any]] = {}  # run_dir -> {trainer, ckpt_mgr, callbacks, fingerprint, model_sig}

# ما يُعدّ «حالة تشغيل سابقة» داخل run_dir (تُنقل كلها معًا إلى _archive عند البدء من جديد)
STATE_ENTRIES = ("checkpoint", "meta.json", "best.weights.h5", "best_meta.json", "best.prev.weights.h5")


def has_saved_state(directory: Optional[str]) -> bool:
    """هل يوجد checkpoint فعلي (لا مجرد مجلد فارغ) في هذا المسار؟"""
    if not directory:
        return False
    return tf.train.latest_checkpoint(os.path.join(directory, "checkpoint")) is not None


def archive_state(directory: str, tag: str) -> str:
    """ينقل (لا يحذف) كل حالة التشغيل السابقة إلى directory/_archive/<tag>/ — قابلة للاسترجاع يدويًا."""
    dest = os.path.join(directory, "_archive", tag)
    os.makedirs(dest, exist_ok=True)
    for name in STATE_ENTRIES:
        src = os.path.join(directory, name)
        if os.path.exists(src):
            shutil.move(src, os.path.join(dest, name))
    return dest


class CheckpointManager:
    """يغلّف tf.train.Checkpoint/CheckpointManager + ملف meta.json صغير للحالات القابلة للتسلسل"""

    def __init__(self, trainer: "GenericTrainer", run_dir: str, max_to_keep: int = 3):
        self.trainer = trainer
        self.run_dir = run_dir
        mount_drive_if_needed(run_dir)
        os.makedirs(run_dir, exist_ok=True)

        self.ckpt_dir = os.path.join(run_dir, "checkpoint")
        self.meta_path = os.path.join(run_dir, "meta.json")

        # tf.train.Checkpoint(trainer=trainer) يتتبّع trainer بالكامل بشكل متداخل:
        # trainer.model, trainer.optimizer (بعد compile), trainer.scheduled_vars,
        # trainer.loss_weights, trainer.uncertainty_layer, trainer.ckpt_epoch — كل شيء دفعة واحدة.
        self.checkpoint = tf.train.Checkpoint(trainer=trainer)
        self.manager = tf.train.CheckpointManager(self.checkpoint, self.ckpt_dir, max_to_keep=max_to_keep)

    def has_checkpoint(self) -> bool:
        return self.manager.latest_checkpoint is not None

    def restore(self) -> Tuple[int, dict]:
        """يُستدعى **بعد** build_trainer_variables() وفقط إن وُجد checkpoint. يُعيد (initial_epoch, callback_states)"""
        latest = self.manager.latest_checkpoint
        status = self.checkpoint.restore(latest)
        status.assert_existing_objects_matched()  # يفشل بوضوح إن تغيّرت بنية config/النموذج بشكل غير متوافق

        meta = {}
        if os.path.exists(self.meta_path):
            try:
                with open(self.meta_path, encoding="utf-8") as f:
                    meta = json.load(f)
            except Exception as e:
                print(f"⚠️ تعذّرت قراءة meta.json ({e}) — الأوزان والـ optimizer سليمة لكن حالة الكولباكس (best/wait) ستبدأ من جديد")

        initial_epoch = int(self.trainer.ckpt_epoch.numpy())
        print(f"✅ تم استرجاع الحالة الكاملة (أوزان + optimizer + جداول) من: {latest}")
        print(f"   الاستئناف من Epoch {initial_epoch + 1}")
        return initial_epoch, meta.get("callback_states", {})

    def save(self, epoch: int, callback_states: Optional[dict] = None):
        self.trainer.ckpt_epoch.assign(epoch)
        path = self.manager.save()
        atomic_write_json(self.meta_path, {
            "epoch": epoch, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "callback_states": callback_states or {},
        })
        return path


def build_trainer_variables(trainer: "GenericTrainer", sample_batch: Tuple[Any, ...]):
    """ينشئ كل متغيرات trainer (النموذج + slots وعدّاد الـ optimizer + متغيرات EMA) **دون أي خطوة تدريب**.

    لماذا: tf.train.Checkpoint.restore يحتاج المتغيرات موجودة، وslots الـ optimizer لا تُخلق إلا عند build().
    الطريقة القديمة كانت تنفّذ train_on_batch ثم تُعيد الأوزان القابلة للتدريب فقط — فتبقى آثارها في:
    optimizer.iterations (=1) وعزوم Adam غير الصفرية، وإحصاءات BatchNormalization المتحركة، وسلسلة RNG.
    هنا: استدعاء أمامي واحد بـ training=False (لا يحدّث BatchNorm) + optimizer.build(...) (يخلق ولا يحدّث)."""
    x_sample = sample_batch[0]
    trainer.model(x_sample, training=False)
    trainer.optimizer.build(trainer.trainable_variables)


def stage_warm_start_weights(config: dict) -> Tuple[str, int, int]:
    """يتحقق من إعدادات warm_start وينسخ ملف الأوزان إلى مكان مؤقت — قبل أرشفة run_dir (قد يكون الملف بداخله)."""
    ws = config["run"].get("warm_start") or {}
    path, epochs_done = ws.get("weights_path"), ws.get("epochs_done")
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"❌ warm_start: run.warm_start.weights_path غير موجود: {path!r}")
    if epochs_done is None or int(epochs_done) < 0:
        raise ValueError("❌ warm_start: يجب تحديد run.warm_start.epochs_done (كم حقبة تدرّبها هذا النموذج فعلًا). "
                         "لا نخمّنه: تخمين 0 يعيد جداول العقوبات إلى الصفر — وهي المشكلة التي وُجد هذا الوضع لحلّها.")
    staged = os.path.join(tempfile.mkdtemp(prefix="warm_start_"), "staged.weights.h5")
    shutil.copy(path, staged)
    return staged, int(epochs_done), int(ws.get("lr_rewarmup_epochs", 0))


In [ ]:
# @title 7.2) DriveMirror (اختياري) + EpochCheckpointCallback + EpochGuard
class DriveMirror(tf.keras.callbacks.Callback):
    """نسخ دوري لمجلد التشغيل بالكامل إلى مسار دائم (مثل Drive) — طبقة أمان إضافية اختيارية.
    يعمل كنسخ مباشر لشجرة مجلد واحدة (بلا أي دمج/ترقيم منفصل)، لذا لا يوجد أي خطر تعارض."""

    def __init__(self, source_dir: str, mirror_dir: str, every: int = 1, verbose: int = 1):
        super().__init__()
        self.source_dir = source_dir
        self.mirror_dir = mirror_dir
        self.every = max(int(every), 1)
        self.verbose = verbose

    def _mirror(self):
        mount_drive_if_needed(self.mirror_dir)
        shutil.copytree(self.source_dir, self.mirror_dir, dirs_exist_ok=True)
        if self.verbose:
            print(f"☁️ [Mirror] {self.source_dir} → {self.mirror_dir}")

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every == 0:
            self._mirror()

    def on_train_end(self, logs=None):
        self._mirror()   # النسخة النهائية (بعد استرجاع أفضل أوزان وإعادة حفظ الـ checkpoint)


class EpochCheckpointCallback(tf.keras.callbacks.Callback):
    """يحفظ الحالة الكاملة في نهاية كل N حقبة، ويجمع حالة أي كولباك يدعم get_state().

    - عدّاد الحقب المنجزة (trainer.ckpt_epoch) يُحدَّث في **كل** حقبة، لا فقط عند الحفظ — ليكون المرجع الصادق
      لـ initial_epoch حتى لو كان save_every > 1.
    - on_train_end يُعيد حفظ الحالة: بعد fit() يكون النموذج في الذاكرة قد تغيّر (تبديل أوزان EMA من Keras، ثم
      استرجاع أفضل أوزان) — فنحفظ الحالة النهائية كي يطابق ما على القرص ما في الذاكرة، ولا تختلف «المتابعة في
      نفس الجلسة» عن «الاستئناف في جلسة جديدة»."""

    def __init__(self, checkpoint_manager: CheckpointManager, stateful_callbacks: Optional[list] = None,
                 save_every: int = 1, verbose: int = 1):
        super().__init__()
        self.ckpt_mgr = checkpoint_manager
        self.stateful_callbacks = stateful_callbacks or []
        self.save_every = max(int(save_every), 1)
        self.verbose = verbose

    def _collect_states(self) -> dict:
        return {cb.__class__.__name__: cb.get_state() for cb in self.stateful_callbacks if hasattr(cb, "get_state")}

    def on_epoch_end(self, epoch, logs=None):
        self.ckpt_mgr.trainer.ckpt_epoch.assign(epoch + 1)
        if (epoch + 1) % self.save_every != 0:
            return
        path = self.ckpt_mgr.save(epoch + 1, self._collect_states())
        if self.verbose:
            print(f"💾 [Checkpoint] Epoch {epoch + 1} → {path}")

    def on_train_end(self, logs=None):
        done = int(self.ckpt_mgr.trainer.ckpt_epoch.numpy())
        if done > 0:
            path = self.ckpt_mgr.save(done, self._collect_states())
            if self.verbose:
                print(f"💾 [Checkpoint] الحالة النهائية (Epoch {done}) → {path}")


class EpochGuard(tf.keras.callbacks.Callback):
    """يمنع الخطأ الصامت الذي يُصفّر جداول العقوبات: إن مرّرتَ initial_epoch لا يطابق عدد الحقب التي أنجزها
    المدرّب فعلًا (نسيته فصار 0 مثلًا) فإن ParamScheduler وجدول lr سيحسبان قيمهما لحقبة خاطئة — أي
    lambda_reg=0 وlambda_calib=0 وتسخين lr من جديد فوق نموذج متقارب."""

    def __init__(self):
        super().__init__()
        self._checked = False

    def on_train_begin(self, logs=None):
        self._checked = False

    def on_epoch_begin(self, epoch, logs=None):
        if self._checked:
            return
        self._checked = True
        done = int(self.model.ckpt_epoch.numpy())
        if epoch != done:
            raise RuntimeError(
                f"❌ initial_epoch={epoch} لكن المدرّب أنجز {done} حقبة فعلًا. بهذا الشكل ستُحسب جداول العقوبات "
                f"وlr للحقبة {epoch} بدل {done} (قد تعود lambda_reg وlambda_calib إلى الصفر). "
                f"استخدم initial_epoch={done} (القيمة العائدة من build_training_system)، أو ابدأ من جديد عبر "
                f"train_mode='new'، أو عطّل الحارس عمدًا بـ config['run']['strict_epoch_guard']=False.")


## 8) `build_training_system` — نقطة الدخول الوحيدة

استدعِها دائمًا **بنفس الطريقة**، سواء كانت هذه أول مرة تدريب أو استئنافًا
بعد انقطاع — الدالة تكتشف ذلك تلقائيًا وتتصرف بالشكل الصحيح دون أي تدخّل منك.

```python
trainer, callbacks, initial_epoch = build_training_system(model_builder_fn, config, sample_batch)
trainer.fit(train_ds, validation_data=val_ds,
            initial_epoch=initial_epoch, epochs=config["run"]["epochs"], callbacks=callbacks)
```

- `model_builder_fn`: دالة **بلا مدخلات** تُرجع نموذج Keras جديد (نفس البنية
  دائمًا في كل استدعاء — هذا شرط أساسي لصحّة الاستئناف).
- `config`: ناتج `build_config(...)`.
- `sample_batch`: دفعة واحدة حقيقية `(x, y)` من بيانات التدريب (لإجبار بناء
  كل المتغيرات قبل أي استرجاع — انظر القسم 7).

In [ ]:
# @title 8) build_optimizer + build_training_system
def build_optimizer(opt_cfg: dict) -> tf.keras.optimizers.Optimizer:
    common = dict(
        learning_rate=opt_cfg["lr_initial"],
        global_clipnorm=opt_cfg.get("clip_norm"),
        use_ema=opt_cfg.get("use_ema", False),
        ema_momentum=opt_cfg.get("ema_momentum", 0.999),
    )
    if opt_cfg.get("name", "adamw") == "adamw":
        opt = tf.keras.optimizers.AdamW(weight_decay=opt_cfg.get("weight_decay", 0.0), **common)
    else:
        opt = tf.keras.optimizers.Adam(**common)

    # ⚙️ Mixed precision: إن كانت السياسة العامة لـ Keras تستخدم float16 للحساب، نغلّف الـ optimizer
    # تلقائيًا بـ LossScaleOptimizer: إلزامي مع mixed_float16 لمنع اختفاء (underflow) التدرجات الصغيرة جدًا.
    policy = tf.keras.mixed_precision.global_policy()
    if policy.compute_dtype == "float16":
        opt = tf.keras.mixed_precision.LossScaleOptimizer(opt)
    return opt


def _resolve_train_mode(train_mode: str, run_dir: str) -> str:
    if train_mode != "auto":
        return train_mode
    return "resume" if has_saved_state(run_dir) else "new"


def build_training_system(
    model_builder_fn: Callable[[], tf.keras.Model],
    config: dict,
    sample_batch: Tuple[Any, Any],
):
    """نقطة الدخول الوحيدة. تُستدعى بنفس الطريقة دائمًا؛ السلوك يُحدَّده config['run']['train_mode']:

      "auto" (افتراضي)  : resume إن وُجدت حالة سابقة في run_dir، وإلا new — هذا هو السلوك القديم بالضبط.
      "new"              : تجاهل أي حالة سابقة (تُؤرشَف تلقائيًا في run_dir/_archive/<timestamp>/، لا تُحذف).
      "resume"           : يفشل بوضوح إن لم توجد حالة — بدل أن يبدأ صامتًا من جديد بغيابها بالخطأ.
      "warm_start"       : يحمّل أوزان model فقط من config['run']['warm_start']['weights_path'] (optimizer/
                           الجداول جديدة)، ويستأنف جداول العقوبات من epochs_done بدل الصفر — هذا يحل مباشرة
                           مشكلة «عقوبات الثقة تبدأ من صفر عند إكمال التدريب فتفسد الموثوقية».
    """
    run = config["run"]
    run_dir = run["run_dir"]
    mirror_dir = run.get("mirror_dir")
    train_mode = run["train_mode"]
    mount_drive_if_needed(run_dir)

    fingerprint = config_fingerprint(config)

    # ── إعادة استخدام نفس الكائنات إن كانت موجودة بالفعل في هذه الجلسة، وبنفس البصمة ──
    cached = TRAINER_REGISTRY.get(run_dir)
    if cached is not None:
        if cached["fingerprint"] != fingerprint:
            print("♻️ config تغيّر منذ آخر بناء لهذا run_dir في هذه الجلسة — تجاهل الكائن المخزَّن وإعادة البناء")
        else:
            print("♻️ تم العثور على مدرّب قائم لهذا الـ run_dir (نفس config) في نفس الجلسة — إعادة استخدامه")
            trainer = cached["trainer"]
            return trainer, cached["callbacks"], int(trainer.ckpt_epoch.numpy())

    # ── استعادة نسخة احتياطية كاملة إن كان run_dir فارغًا وmirror_dir يحوي بيانات ──
    if mirror_dir:
        mount_drive_if_needed(mirror_dir)
        if not has_saved_state(run_dir) and has_saved_state(mirror_dir):
            print(f"📥 استعادة نسخة احتياطية كاملة: {mirror_dir} → {run_dir}")
            shutil.copytree(mirror_dir, run_dir, dirs_exist_ok=True)

    train_mode = _resolve_train_mode(train_mode, run_dir)
    is_resuming = (train_mode == "resume")

    if train_mode == "resume" and not has_saved_state(run_dir):
        raise FileNotFoundError(
            f"❌ run.train_mode='resume' لكن لا يوجد checkpoint في {run_dir}. "
            f"استخدم train_mode='new' للبدء من الصفر عمدًا، أو 'auto' ليختار الإطار تلقائيًا.")

    warm_start_staged = warm_start_epochs_done = warm_rewarm_epochs = None
    if train_mode == "warm_start":
        warm_start_staged, warm_start_epochs_done, warm_rewarm_epochs = stage_warm_start_weights(config)

    if train_mode in ("new", "warm_start") and has_saved_state(run_dir):
        if run["on_existing"] == "error":
            raise FileExistsError(
                f"❌ توجد حالة تدريب سابقة في {run_dir} وrun.on_existing='error'. "
                f"غيّرها إلى 'archive' للسماح بنقلها تلقائيًا، أو استخدم run_dir جديدًا.")
        tag = time.strftime("%Y%m%d_%H%M%S")
        dest = archive_state(run_dir, tag)
        print(f"🗄️ أُرشِفت حالة التشغيل السابقة إلى: {dest} (لم تُحذف)")

    tf.keras.utils.set_random_seed(run.get("seed", 42))

    base_model = model_builder_fn()
    if train_mode == "warm_start":
        base_model.load_weights(warm_start_staged)
        shutil.rmtree(os.path.dirname(warm_start_staged), ignore_errors=True)
        print(f"🌱 [warm_start] أوزان النموذج حُمِّلت من نسخة أُنجزت {warm_start_epochs_done} حقبة — "
              f"optimizer جديد بالكامل، والجداول ستستأنف من هذه الحقبة (لا من الصفر)")

    trainer = GenericTrainer(base_model, config)
    lr_fn = build_lr_schedule_fn(
        config["optimizer"],
        rewarm_from_epoch=warm_start_epochs_done if train_mode == "warm_start" else None,
        rewarm_epochs=warm_rewarm_epochs or 0,
    )
    trainer.compile(
        optimizer=build_optimizer(config["optimizer"]),
        jit_compile=bool(config["optimizer"].get("use_xla", False)),
    )

    ckpt_mgr = CheckpointManager(trainer, run_dir=run_dir, max_to_keep=config["checkpoint"].get("max_to_keep", 3))

    # ── بناء كل متغيرات trainer (نموذج + optimizer) بلا أي خطوة تدريب (انظر القسم 7.1) ──
    build_trainer_variables(trainer, sample_batch)

    if is_resuming:
        initial_epoch, callback_states = ckpt_mgr.restore()
    else:
        initial_epoch, callback_states = 0, {}
        if train_mode == "warm_start":
            initial_epoch = warm_start_epochs_done
            trainer.ckpt_epoch.assign(warm_start_epochs_done)
            apply_schedules(trainer, config, warm_start_epochs_done)   # الجداول تبدأ من هنا لا من الصفر
        print(f"🆕 بدء تدريب {'دافئ (warm_start)' if train_mode == 'warm_start' else 'جديد'} — run_dir: {run_dir}")

    # ═══════════════════════════════ بناء الكولباكس من config بالكامل ═══════════════════════════════
    callbacks = []
    verbose = run.get("verbose", 1)

    for pname, pcfg in config["loss"].get("schedules", {}).items():
        callbacks.append(ParamScheduler(
            trainer.scheduled_vars[pname], start=pcfg["start"], end=pcfg["end"],
            warmup_epochs=pcfg.get("warmup_epochs", 0), schedule=pcfg.get("schedule", "linear"),
            label=pname, verbose=verbose,
        ))

    callbacks.append(tf.keras.callbacks.LearningRateScheduler(lr_fn, verbose=0))

    if run.get("strict_epoch_guard", True):
        callbacks.append(EpochGuard())

    es_cfg = config["callbacks"]["early_stopping"]
    best_path = os.path.join(run_dir, "best.weights.h5") if config["checkpoint"].get("save_best_weights", True) else None
    best_tracker = BestModelTracker(
        base_model=base_model, best_path=best_path,
        monitor=es_cfg["monitor"], mode=es_cfg.get("mode", "min"),
        patience=es_cfg["patience"], min_delta=es_cfg.get("min_delta", 1e-4),
        smoothing=es_cfg.get("smoothing", "window"), window=es_cfg.get("smoothing_window", 3),
        ema_beta=es_cfg.get("ema_beta", 0.7), restore_best_weights=es_cfg.get("restore_best_weights", True),
        verbose=verbose, initial_state=callback_states.get("BestModelTracker"),
        trainer=trainer, weights_snapshot=es_cfg.get("weights_snapshot", "raw"),
    )
    callbacks.append(best_tracker)
    stateful_callbacks = [best_tracker]

    tw_freq = config["callbacks"].get("task_weight_update_frequency", 0)
    if tw_freq and not trainer.use_uncertainty_weighting:
        callbacks.append(TaskWeightUpdater(trainer, update_frequency=tw_freq, verbose=verbose))

    metrics_logger = MetricsLogger(
        log_every=config["callbacks"].get("metrics_log_every", 5), verbose=verbose,
        initial_state=callback_states.get("MetricsLogger"),
        class_baselines=config["callbacks"].get("class_baselines"),
    )
    callbacks.append(metrics_logger)
    stateful_callbacks.append(metrics_logger)

    snap_cfg = config["callbacks"].get("snapshots", {})
    if snap_cfg.get("epochs"):
        callbacks.append(SnapshotEnsemble(
            save_epochs=snap_cfg["epochs"], save_dir=os.path.join(run_dir, snap_cfg.get("dir", "snapshots")),
            base_model=base_model, verbose=verbose,
        ))

    # الحفظ الكامل للحالة (آخر كولباك يضيف حالة، وقبل DriveMirror حتى تُنسخ أحدث نسخة)
    callbacks.append(EpochCheckpointCallback(
        ckpt_mgr, stateful_callbacks=stateful_callbacks,
        save_every=config["checkpoint"].get("save_every", 1), verbose=verbose,
    ))

    if mirror_dir:
        callbacks.append(DriveMirror(run_dir, mirror_dir, every=run.get("mirror_every", 1), verbose=verbose))

    TRAINER_REGISTRY[run_dir] = {
        "trainer": trainer, "ckpt_mgr": ckpt_mgr, "callbacks": callbacks,
        "fingerprint": fingerprint, "model_sig": model_signature(base_model),
    }

    print(f"\n{'=' * 70}\n🚀 نظام التدريب جاهز")
    print(f"   وضع البدء: {train_mode}")
    print(f"   الأهداف: {trainer.target_names}")
    print(f"   الاستئناف: {'نعم، من Epoch ' + str(initial_epoch + 1) if is_resuming else ('لا، تدريب جديد' if train_mode != 'warm_start' else f'لا — بدء دافئ من حقبة {initial_epoch}')}")
    print(f"   موازنة المهام التلقائية: {trainer.use_uncertainty_weighting}"
          + (f" (Kendall على: {trainer.kendall_task_names} | وزن ثابت: {trainer.fixed_task_names})"
             if trainer.use_uncertainty_weighting else ""))
    print(f"   المعاملات المجدولة: {list(trainer.scheduled_vars.keys())}")
    print(f"   اختيار الأفضل: monitor={es_cfg['monitor']} mode={es_cfg.get('mode','min')} smoothing={es_cfg.get('smoothing','window')}")
    print(f"{'=' * 70}\n")

    return trainer, callbacks, initial_epoch


## 9) اختبار تحقّق فعلي (Smoke Test)

شغّل هذه الخلية للتأكد **بنفسك** أن مشكلة "فقدان زخم الـ optimizer عند
الاستئناف" قد حُلّت فعليًا، قبل استخدام الإطار على بياناتك الحقيقية. الاختبار:

1. يبني نموذجًا صغيرًا وهميًا بهدفين مختلفي النوع (`evidential` + `classification`)
   لإظهار مرونة الإطار مع مهام متعددة ومختلطة.
2. يدرّبه حقبتين، ثم يأخذ لقطة من قيم كل متغيرات الـ optimizer (بما فيها
   *slots* الزخم).
3. **يحذف المدرّب بالكامل من الذاكرة** (محاكاة انقطاع Colab).
4. يعيد بناء مدرّب جديد الكائن من الصفر لنفس `run_dir`، ويتأكد أن:
   - رقم الحقبة المستأنفة صحيح (`initial_epoch == 2`).
   - **كل** قيم متغيرات الـ optimizer مطابقة تمامًا لما كانت قبل "الانقطاع".
5. يستكمل التدريب حقبتين إضافيتين للتأكد أن كل شيء يعمل بسلاسة بعد الاستئناف.
6. **🆕 يتحقق أيضًا من `warm_start`**: يُدرِّب نموذجًا «منتهيًا» حتى تصل جداول العقوبات لقيمتها
   النهائية، ثم يستكمله بـ `train_mode='warm_start'` ويتأكد أن الجداول تُستأنف من تلك القيمة
   **فورًا** لا من الصفر — عكس ما يحدث مع `load_weights` بسيطة.

In [ ]:
# @title 9) Smoke Test — يثبت الحفاظ على حالة الـ optimizer + إصلاحات هذه النسخة
def _dummy_model_builder():
    inp = tf.keras.Input(shape=(8,), name="features")
    h = tf.keras.layers.Dense(16, activation="relu")(inp)

    mu = tf.keras.layers.Dense(1, name="y_a", dtype='float32')(h)
    nu = tf.keras.layers.Dense(1, activation="softplus", name="y_a_nu", dtype='float32')(h)
    alpha = tf.keras.layers.Dense(1, activation="softplus", name="y_a_alpha", dtype='float32')(h)
    beta = tf.keras.layers.Dense(1, activation="softplus", name="y_a_beta", dtype='float32')(h)
    conf = tf.keras.layers.Dense(1, activation="sigmoid", name="y_a_confidence", dtype='float32')(h)
    logits_b = tf.keras.layers.Dense(1, activation="sigmoid", name="y_b_logits", dtype='float32')(h)

    outputs = {
        "y_a": mu, "y_a_nu": nu, "y_a_alpha": alpha, "y_a_beta": beta, "y_a_confidence": conf,
        "y_b_logits": logits_b,
    }
    return tf.keras.Model(inputs=inp, outputs=outputs)


smoke_config = build_config({
    "run": {"run_dir": "/content/_smoke_test_run", "epochs": 4, "batch_size": 32, "verbose": 1},
    "targets": {
        "a": {
            "true_key": "y_a", "task_type": "evidential",
            "output_keys": {"mu": "y_a", "nu": "y_a_nu", "alpha": "y_a_alpha",
                             "beta": "y_a_beta", "confidence": "y_a_confidence"},
            "use_calibration_loss": True, "lambda_reg_var": "lambda_reg", "lambda_calib_var": "lambda_calib",
        },
        "b": {"true_key": "y_b", "task_type": "classification", "output_keys": {"logits": "y_b_logits"}, "binary": True},
    },
    "loss": {
        "use_uncertainty_weighting": True,
        "schedules": {
            "lambda_reg": {"start": 0.0, "end": 0.05, "warmup_epochs": 2, "schedule": "linear"},
            "lambda_calib": {"start": 0.0, "end": 0.1, "warmup_epochs": 2, "schedule": "cosine"},
        },
    },
    "optimizer": {"lr_initial": 1e-3, "lr_warmup_epochs": 0, "lr_schedule": {"type": "constant"}},
    "callbacks": {"early_stopping": {"patience": 100}, "metrics_log_every": 1},
})

shutil.rmtree(smoke_config["run"]["run_dir"], ignore_errors=True)

_rng = np.random.default_rng(0)
_N = 256
_X = _rng.normal(size=(_N, 8)).astype("float32")
_y_a = _rng.normal(size=(_N, 1)).astype("float32")
_y_b = _rng.integers(0, 2, size=(_N, 1)).astype("float32")
smoke_train_ds = tf.data.Dataset.from_tensor_slices((_X, {"y_a": _y_a, "y_b": _y_b})).batch(32, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
smoke_sample_batch = next(iter(smoke_train_ds))

print("── المرحلة 1: تدريب من الصفر لحقبتين ──")
trainer1, callbacks1, initial_epoch1 = build_training_system(_dummy_model_builder, smoke_config, smoke_sample_batch)
trainer1.fit(smoke_train_ds, initial_epoch=initial_epoch1, epochs=2, callbacks=callbacks1, verbose=1)

opt_values_before = [v.numpy().copy() for v in opt_variables(trainer1.optimizer)]

print("\n── محاكاة انقطاع Colab: مسح كل شيء من الذاكرة ──")
del trainer1, callbacks1
TRAINER_REGISTRY.pop(smoke_config["run"]["run_dir"], None)
tf.keras.backend.clear_session()

print("\n── المرحلة 2: إعادة بناء المدرّب من الصفر (Python) + استئناف من الـ checkpoint ──")
trainer2, callbacks2, initial_epoch2 = build_training_system(_dummy_model_builder, smoke_config, smoke_sample_batch)
assert initial_epoch2 == 2, f"❌ initial_epoch متوقع=2 لكن الفعلي={initial_epoch2}"

opt_values_after = [v.numpy() for v in opt_variables(trainer2.optimizer)]
assert len(opt_values_before) == len(opt_values_after), "❌ عدد متغيرات الـ optimizer غير متطابق!"
for a, b in zip(opt_values_before, opt_values_after):
    np.testing.assert_allclose(a, b, rtol=1e-5, atol=1e-6)
print(f"✅ تم التحقق: كل متغيرات الـ optimizer ({len(opt_values_after)} متغيرًا، بما فيها الزخم) استُرجعت بدقة كاملة")

print("\n── المرحلة 3: استكمال التدريب حقبتين إضافيتين (3 و 4) ──")
trainer2.fit(smoke_train_ds, initial_epoch=initial_epoch2, epochs=4, callbacks=callbacks2, verbose=1)

print("\n🎉 الاختبار الأصلي نجح — الإطار يحافظ على حالة التدريب الكاملة عبر الاستئناف")

# ── 🆕 اختبار warm_start: عقوبات الثقة تستأنف من حقبتها، لا من الصفر ──
print(f"\n{'=' * 70}\n── اختبار warm_start ──\n{'=' * 70}")
warm_run_dir = "/content/_smoke_test_warm_start"
shutil.rmtree(warm_run_dir, ignore_errors=True)
finished_cfg = deep_update(smoke_config, {"run": {"run_dir": warm_run_dir, "epochs": 6}})
finished_trainer, finished_cbs, ie = build_training_system(_dummy_model_builder, finished_cfg, smoke_sample_batch)
finished_trainer.fit(smoke_train_ds, initial_epoch=ie, epochs=6, callbacks=finished_cbs, verbose=0)
lambda_reg_at_end = float(finished_trainer.scheduled_vars["lambda_reg"].numpy())
assert abs(lambda_reg_at_end - 0.05) < 1e-6, f"❌ توقعنا lambda_reg=0.05 بعد اكتمال warmup لكن الفعلي={lambda_reg_at_end}"
weights_path = "/content/_smoke_finished.weights.h5"
finished_trainer.model.save_weights(weights_path)

TRAINER_REGISTRY.pop(warm_run_dir, None)
tf.keras.backend.clear_session()

warm_cfg = deep_update(smoke_config, {
    "run": {"run_dir": "/content/_smoke_test_warm_start_2", "epochs": 8, "train_mode": "warm_start",
            "warm_start": {"weights_path": weights_path, "epochs_done": 6, "lr_rewarmup_epochs": 1}},
})
shutil.rmtree(warm_cfg["run"]["run_dir"], ignore_errors=True)
warm_trainer, warm_cbs, warm_ie = build_training_system(_dummy_model_builder, warm_cfg, smoke_sample_batch)
lambda_reg_after_warm_start = float(warm_trainer.scheduled_vars["lambda_reg"].numpy())
assert warm_ie == 6, f"❌ توقعنا initial_epoch=6 لكن الفعلي={warm_ie}"
assert abs(lambda_reg_after_warm_start - 0.05) < 1e-6, (
    f"❌ warm_start فشل في استئناف الجدول: lambda_reg={lambda_reg_after_warm_start} (توقعنا 0.05، لا 0.0)")
print(f"✅ تم التحقق: warm_start استأنف lambda_reg={lambda_reg_after_warm_start:.4f} من الحقبة {warm_ie} "
      f"(لا 0.0000 كما يحدث مع load_weights بسيطة)")
warm_trainer.fit(smoke_train_ds, initial_epoch=warm_ie, epochs=8, callbacks=warm_cbs, verbose=0)
print("🎉 اختبار warm_start نجح — التدريب استكمل بلا أي انهيار في عقوبات الثقة")


## 10) قالب استخدام كامل على مشروعك الحقيقي

هذا القالب **مثال فقط** لإظهار المرونة (أهداف `high`/`low`/`close` من نوع
`evidential` + قيد منطقي بينها + هدف `direction` من نوع `classification`).
**أسماء الأهداف هنا ليست جزءًا من الإطار** — احذفها وضع أسماء أهدافك الحقيقية
(قد يكون هدفًا واحدًا فقط، أو خمسين هدفًا، بأي أسماء تريد).

عدّل الخلية التالية بالكامل حسب مشروعك: `real_model_builder`، `real_config`،
والبيانات (`real_train_ds` / `real_val_ds`).

In [ ]:
# @title 10.1) مثال: بناء النموذج (بدّله ببنيتك الحقيقية)
def real_model_builder():
    """
    مثال بمدخل واحد فقط لغرض التبسيط. الإطار يدعم مدخلات متعددة بالتساوي —
    فقط مرّر tuple/dict من tf.keras.Input إلى inputs=[...] كما يفعل Keras عادة،
    فالإطار لا يفتح x إطلاقًا (يمرره لـ self.model(x) كما هو).
    """
    inp = tf.keras.Input(shape=(32,), name="features")
    h = tf.keras.layers.Dense(64, activation="relu")(inp)
    h = tf.keras.layers.Dense(64, activation="relu")(h)

    def evidential_head(prefix):
        # dtype='float32' صريح لكل مخرجات NIG — أمان إلزامي إن فعّلت mixed precision لاحقًا
        # (انظر التحذير في القسم 1.1)؛ بلا ضرر إطلاقًا إن كانت السياسة العامة float32 أصلًا.
        mu = tf.keras.layers.Dense(1, name=f"y_{prefix}", dtype='float32')(h)
        nu = tf.keras.layers.Dense(1, activation="softplus", name=f"y_{prefix}_nu", dtype='float32')(h)
        alpha = tf.keras.layers.Dense(1, activation="softplus", name=f"y_{prefix}_alpha", dtype='float32')(h)
        beta = tf.keras.layers.Dense(1, activation="softplus", name=f"y_{prefix}_beta", dtype='float32')(h)
        conf = tf.keras.layers.Dense(1, activation="sigmoid", name=f"y_{prefix}_confidence", dtype='float32')(h)
        return mu, nu, alpha, beta, conf

    outputs = {}
    for t in ["high", "low", "close"]:
        mu, nu, alpha, beta, conf = evidential_head(t)
        outputs.update({f"y_{t}": mu, f"y_{t}_nu": nu, f"y_{t}_alpha": alpha,
                         f"y_{t}_beta": beta, f"y_{t}_confidence": conf})

    outputs["y_direction_logits"] = tf.keras.layers.Dense(1, activation="sigmoid", name="y_direction_logits", dtype='float32')(h)

    return tf.keras.Model(inputs=inp, outputs=outputs)


# ── مثال على قيد منطقي اختياري بين المخرجات (high >= low, close بين الاثنين) ──
def high_low_close_order_penalty(outputs):
    h, l, c = outputs["y_high"], outputs["y_low"], outputs["y_close"]
    viol_hl = tf.nn.relu(l - h)
    viol_hc = tf.nn.relu(c - h)
    viol_lc = tf.nn.relu(l - c)
    return tf.reduce_mean(viol_hl + viol_hc + viol_lc)


In [ ]:
# @title 10.2) مثال: قاموس الإعدادات الكامل لمشروعك
real_config = build_config({
    "run": {
        "run_dir": "/content/drive/MyDrive/training_runs/my_project_v1",  # ⚠️ غيّره لمسارك
        "mirror_dir": None,
        "epochs": 60,
        "batch_size": 64,
        "seed": 42,
        "verbose": 1,
        # 🆕 وضع البدء — اتركه "auto" في الاستخدام العادي (نفس السلوك القديم: يستأنف إن وُجدت حالة).
        # لإكمال تدريب نموذج *لم يُدرَّب عبر هذا الإطار* (مثل حالتك — نموذج منتهٍ محفوظ بـ save_weights
        # بسيطة) استخدم warm_start كما في القسم 11 أدناه، حتى لا تبدأ عقوبات الثقة من الصفر مجددًا.
        "train_mode": "auto",   # auto | new | resume | warm_start
    },
    "targets": {
        "high": {
            "true_key": "y_high", "task_type": "evidential",
            "output_keys": {"mu": "y_high", "nu": "y_high_nu", "alpha": "y_high_alpha",
                             "beta": "y_high_beta", "confidence": "y_high_confidence"},
            "use_calibration_loss": True, "lambda_reg_var": "lambda_reg", "lambda_calib_var": "lambda_calib",
            # 🆕 خيارات اختيارية (معلَّقة) — فعّلها فقط بعد تجربتها على High أولًا ومقارنة val_mae:
            # "normalize_reg": True,      # Meinert et al. 2023 — يقلل تسرّب aleatoric إلى epistemic
            # "beta_nll": 0.5,            # امتداد تجريبي غير موثَّق رسميًا لـ NIG — قيّمه بنفسك
            # "huber_weight": 0.05,
        },
        "low": {
            "true_key": "y_low", "task_type": "evidential",
            "output_keys": {"mu": "y_low", "nu": "y_low_nu", "alpha": "y_low_alpha",
                             "beta": "y_low_beta", "confidence": "y_low_confidence"},
            "use_calibration_loss": True, "lambda_reg_var": "lambda_reg", "lambda_calib_var": "lambda_calib",
        },
        "close": {
            "true_key": "y_close", "task_type": "evidential",
            "output_keys": {"mu": "y_close", "nu": "y_close_nu", "alpha": "y_close_alpha",
                             "beta": "y_close_beta", "confidence": "y_close_confidence"},
            "use_calibration_loss": True, "lambda_reg_var": "lambda_reg", "lambda_calib_var": "lambda_calib",
        },
        "direction": {
            "true_key": "y_direction", "task_type": "classification",
            "output_keys": {"logits": "y_direction_logits"}, "binary": True, "loss_weight": 0.5,
        },
    },
    "loss": {
        "use_uncertainty_weighting": True,
        "schedules": {
            "lambda_reg":   {"start": 0.0, "end": 0.05, "warmup_epochs": 20, "schedule": "linear"},
            "lambda_calib": {"start": 0.0, "end": 0.15, "warmup_epochs": 20, "schedule": "cosine"},
            "penalty_weight": {"start": 0.0, "end": 0.3, "warmup_epochs": 25, "schedule": "linear"},
        },
        "constraints": [
            {"name": "ohlc_order", "fn": high_low_close_order_penalty, "weight_var": "penalty_weight"},
        ],
    },
    "optimizer": {
        "name": "adamw", "lr_initial": 5e-5, "lr_min": 5e-7, "lr_warmup_epochs": 3,
        "lr_schedule": {"type": "cosine_restarts", "cycle_length": 10, "cycle_mult": 1.5},
        "weight_decay": 1e-4, "clip_norm": 1.0, "use_ema": True, "ema_momentum": 0.999,
    },
    "callbacks": {
        "early_stopping": {
            "monitor": "val_loss", "patience": 15, "min_delta": 1e-4,
            "smoothing": "window", "smoothing_window": 3,   # أو "ema" لتنعيم أسّي بدل نافذة ثابتة
            # 🆕 "ema_weights" يحفظ متوسط EMA (يتطلب optimizer.use_ema=True أعلاه، وهو مفعّل هنا) بدل
            # الأوزان الخام عند كل تحسّن — أكثر استقرارًا للنشر النهائي. اتركها "raw" إن كنت تفضّل
            # الأوزان الخام كما كانت في النسخة السابقة من الإطار.
            "weights_snapshot": "raw",
        },
        "metrics_log_every": 5,
    },
    "checkpoint": {"save_every": 1, "max_to_keep": 3, "save_best_weights": True},
})

print(json.dumps({"targets": list(real_config["targets"].keys()),
                   "schedules": list(real_config["loss"]["schedules"].keys()),
                   "constraints": [c["name"] for c in real_config["loss"]["constraints"]]}, indent=2, ensure_ascii=False))


In [ ]:
# @title 10.3) مثال: تجهيز البيانات + التدريب الفعلي
# ⚠️ بدّل هذا القسم ببياناتك الحقيقية. الشرط الوحيد: y يجب أن يكون dict بمفاتيح
# مطابقة لـ true_key في كل هدف داخل config['targets'].

# --- مثال توضيحي فقط (بيانات وهمية بنفس الشكل) ---
# _rng = np.random.default_rng(123)
# _N = 2000
# _X = _rng.normal(size=(_N, 32)).astype("float32")
# _y = {
#     "y_high": _rng.normal(size=(_N, 1)).astype("float32"),
#     "y_low": _rng.normal(size=(_N, 1)).astype("float32"),
#     "y_close": _rng.normal(size=(_N, 1)).astype("float32"),
#     "y_direction": _rng.integers(0, 2, size=(_N, 1)).astype("float32"),
# }
# _split = int(_N * 0.85)
# ⚡ .cache() يخزّن البيانات في الذاكرة بعد أول قراءة (مفيد إن كانت تُبنى بمعالجة مسبقة
#   مكلفة)، .prefetch(AUTOTUNE) يُحضّر الدفعة التالية أثناء تدريب GPU للدفعة الحالية —
#   يقلّل انتظار GPU لبيانات جديدة، وهو غالبًا العنق الحقيقي حين يبدو GPU "خاملًا".
# drop_remainder=True يضمن شكل دفعة ثابت — مطلوب لتفعيل XLA (use_xla) بأمان.
# real_train_ds = tf.data.Dataset.from_tensor_slices(
#     (_X[:_split], {k: v[:_split] for k, v in _y.items()})
# ).cache().shuffle(1024).batch(real_config["run"]["batch_size"], drop_remainder=True).prefetch(tf.data.AUTOTUNE)
# real_val_ds = tf.data.Dataset.from_tensor_slices(
#     (_X[_split:], {k: v[_split:] for k, v in _y.items()})
# ).cache().batch(real_config["run"]["batch_size"], drop_remainder=True).prefetch(tf.data.AUTOTUNE)
# real_sample_batch = next(iter(real_train_ds))
# --- نهاية المثال التوضيحي ---

# real_trainer, real_callbacks, real_initial_epoch = build_training_system(
#     real_model_builder, real_config, real_sample_batch
# )

# real_history = real_trainer.fit(
#     real_train_ds,
#     validation_data=real_val_ds,
#     initial_epoch=real_initial_epoch,
#     epochs=real_config["run"]["epochs"],
#     callbacks=real_callbacks,
#     verbose=1,
# )


## 11) الاستئناف بعد انقطاع Colab — التعليمات العملية

عندما ينقطع Runtime (انتهاء وقت الجلسة، إعادة تشغيل يدوية، إلخ):

1. أعد تشغيل الدفتر بالكامل من الأعلى (Runtime → Run all)، **بدون أي تعديل**
   على خلية `real_config` أو `real_model_builder` (يجب أن تكون البنية مطابقة
   تمامًا لما كانت عليه قبل الانقطاع؛ يمكنك بالطبع تعديل عدد الحقب
   `config['run']['epochs']` لرفعه إن أردت الاستمرار لفترة أطول).
2. عند تنفيذ خلية `build_training_system(...)` مجددًا، ستطبع الدالة تلقائيًا:
   ```
   ✅ تم استرجاع الحالة الكاملة (أوزان + optimizer + جداول) من: ...
      الاستئناف من Epoch N+1
   ```
   وستعود `initial_epoch` بالقيمة الصحيحة.
3. نفّذ `trainer.fit(..., initial_epoch=initial_epoch, ...)` كما هو — Keras
   سيبدأ تلقائيًا من الحقبة الصحيحة، وكل الكولباكس المجدولة
   (`ParamScheduler`, جدولة معدل التعلّم) ستُحسب قيمها الصحيحة لتلك الحقبة
   تلقائيًا (لأنها دوال محضة في رقم الحقبة).

**الشرط الوحيد لضمان ذلك**: أن يكون `config['run']['run_dir']` مسارًا **دائمًا**
(داخل `/content/drive/MyDrive/...` بعد تركيب Drive) — لا مسارًا محليًا تحت
`/content/` بدون Drive، لأن الأخير يُمسح بالكامل مع كل انقطاع.

إن كنت تفضّل تدريبًا سريعًا محليًا مع نسخة احتياطية دورية على Drive، استخدم:
```python
"run": {
    "run_dir": "/content/_fast_local_run",                              # سريع، يُمسح عند الانقطاع
    "mirror_dir": "/content/drive/MyDrive/training_runs/my_project_v1",  # دائم، يُنسخ كل epoch
    "mirror_every": 1,
}
```
عند إعادة التشغيل بعد انقطاع، سيجد `build_training_system` أن `run_dir`
المحلي فارغ وأن `mirror_dir` يحوي بيانات، فينسخها تلقائيًا قبل الاستمرار.

---

## 11.1) 🆕 إكمال تدريب نموذج مُنجَز مسبقًا (خارج هذا الإطار) — `warm_start`

هذه هي حالتك تحديدًا: نموذج تدرّب سابقًا (ربما بكود آخر، أو بإطار قديم بلا checkpoint كامل) وتريد
تدريبه حقبًا إضافية. المشكلة الأصلية: `load_weights` بسيطة تعني `optimizer` جديد (لا ضرر — الزخم يُبنى
من جديد خلال حقب قليلة) **لكن** أيضًا معاملات مجدولة مثل `lambda_reg`/`lambda_calib` تُعاد إلى قيمة
`start` (عادة 0) بدل قيمتها النهائية — فيتدرّب النموذج حقبة كاملة بعقوبة ثقة معطَّلة فجأة، وهذا يُفسد
الدقة والموثوقية التي بناها سابقًا.

```python
config = build_config({
    "run": {
        "run_dir": "/content/drive/MyDrive/training_runs/my_project_v2",  # مسار جديد لهذه المرحلة
        "epochs": 80,   # العدد الإجمالي المطلوب الوصول إليه (وليس عدد الحقب الإضافية فقط)
        "train_mode": "warm_start",
        "warm_start": {
            "weights_path": "/content/drive/MyDrive/my_old_model.weights.h5",  # ملف الأوزان القديم
            "epochs_done": 60,          # كم حقبة تدرّبها هذا النموذج فعليًا — **حدِّدها بدقة، لا تخمّنها**
            "lr_rewarmup_epochs": 3,    # تسخين قصير لـ lr لأن optimizer جديد بزخم صفري (0 = بلا تسخين)
        },
    },
    # ... بقية config كما في القسم 10.2 (نفس الأهداف، نفس الجداول، إلخ)
})

trainer, callbacks, initial_epoch = build_training_system(real_model_builder, config, real_sample_batch)
# initial_epoch = 60 تلقائيًا، وlambda_reg/lambda_calib تُضبَط فورًا على قيمتها عند الحقبة 60 — لا الصفر.
trainer.fit(real_train_ds, validation_data=real_val_ds,
            initial_epoch=initial_epoch, epochs=config["run"]["epochs"], callbacks=callbacks)
```

بعد هذه المرة الأولى، إن انقطع Colab في منتصف هذه المرحلة، **لا تكرّر `train_mode="warm_start"`** —
غيّره إلى `"auto"` (أو احذفه، فهو الافتراضي): الآن أصبح لديك checkpoint كامل من هذا الإطار في
`run_dir` الجديد، فالاستئناف العادي (البند 11 أعلاه) هو الصحيح.

## 12) (اختياري) تدريب K-Fold مع استئناف كامل لكل Fold على حدة

كل خصوصية بنية بياناتك (مدخلات متعددة، إطارات زمنية متعددة، إلخ) تعيش بالكامل
داخل `dataset_builder_fn` التي تكتبها أنت — الإطار لا يفترض عنها شيئًا. كل
Fold يحصل على `run_dir` مستقل (`{base_run_dir}/fold_{i}`)، وبالتالي يستفيد من
نفس آلية الاستئناف الكاملة: إن انقطع التدريب في منتصف Fold 3 مثلًا، إعادة
تشغيل هذه الخلية تتخطى Fold 1 و 2 (اكتملا فعلًا) وتستأنف Fold 3 من حيث توقف.

In [ ]:
# @title 12) K-Fold Training (اختياري)
def purged_walk_forward_splits(n_samples: int, n_splits: int = 5, purge: int = 50, embargo: int = 50):
    fold_size = n_samples // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        val_start = i * fold_size
        val_end = min((i + 1) * fold_size, n_samples)
        train_end = max(0, val_start - purge)
        splits.append((np.arange(0, train_end), np.arange(val_start, val_end)))
    return splits


def run_kfold_training(
    n_samples: int,
    model_builder_fn: Callable[[], tf.keras.Model],
    config_template: dict,
    dataset_builder_fn: Callable[[np.ndarray, np.ndarray], Tuple[tf.data.Dataset, tf.data.Dataset, Tuple[Any, Any]]],
    n_splits: int = 5, purge: int = 50, embargo: int = 50, min_train: int = 100, min_val: int = 20,
):
    """
    dataset_builder_fn(train_idx, val_idx) -> (train_ds, val_ds, sample_batch)
    """
    splits = purged_walk_forward_splits(n_samples, n_splits, purge, embargo)
    base_run_dir = config_template["run"]["run_dir"]

    trained_models, histories = [], []
    for fold_idx, (train_idx, val_idx) in enumerate(splits, start=1):
        print(f"\n{'=' * 80}\n📂 Fold {fold_idx}/{n_splits} — train={len(train_idx)} val={len(val_idx)}\n{'=' * 80}")
        if len(train_idx) < min_train or len(val_idx) < min_val:
            print("⚠️ بيانات غير كافية — تخطّي هذا الـ Fold")
            continue

        fold_config = copy.deepcopy(config_template)
        fold_config["run"]["run_dir"] = os.path.join(base_run_dir, f"fold_{fold_idx}")

        train_ds, val_ds, sample_batch = dataset_builder_fn(train_idx, val_idx)

        trainer, callbacks, initial_epoch = build_training_system(model_builder_fn, fold_config, sample_batch)
        history = trainer.fit(
            train_ds, validation_data=val_ds, initial_epoch=initial_epoch,
            epochs=fold_config["run"]["epochs"], callbacks=callbacks, verbose=1,
        )

        trained_models.append(trainer.model)
        histories.append(history.history)
        best_val = min(history.history.get("val_loss", [np.nan])) if history.history.get("val_loss") else float("nan")
        print(f"✅ Fold {fold_idx} انتهى — أفضل val_loss: {best_val:.4f}")

    print(f"\n{'=' * 80}\n🏁 انتهى K-Fold: {len(trained_models)} نموذج جاهز للـ Ensemble\n{'=' * 80}")
    return trained_models, histories


## 13) (اختياري) استدلال Ensemble معمَّم لأي هدف evidential

`output_keys` بنفس صيغة `target_cfg['output_keys']` المستخدمة في `config` —
بلا أي افتراض لاسم الهدف. استدعها بـ
`ensemble_predict_evidential(models, x, real_config['targets']['close']['output_keys'])`
مثلًا.

In [ ]:
# @title 13) Ensemble Inference (اختياري)
def ensemble_predict_evidential(models: List[tf.keras.Model], x, output_keys: dict) -> Dict[str, np.ndarray]:
    mus, nus, alphas, betas = [], [], [], []
    for m in models:
        out = m(x, training=False)
        mus.append(out[output_keys["mu"]].numpy())
        nus.append(out[output_keys["nu"]].numpy())
        alphas.append(out[output_keys["alpha"]].numpy())
        betas.append(out[output_keys["beta"]].numpy())

    mus, nus, alphas, betas = (np.stack(a, axis=0) for a in (mus, nus, alphas, betas))

    mu_ensemble = mus.mean(axis=0)
    aleatoric_within = (betas / (alphas - 1.0 + 1e-6)).mean(axis=0)
    epistemic_within = (betas / (nus * (alphas - 1.0) + 1e-6)).mean(axis=0)
    epistemic_between = mus.var(axis=0)  # تباين آراء النماذج المختلفة = عدم يقين حقيقي إضافي

    epistemic_total = epistemic_within + epistemic_between
    total_uncertainty = aleatoric_within + epistemic_total
    confidence = np.clip(1.0 / (1.0 + total_uncertainty), 0.01, 0.99)

    return {
        "mu": mu_ensemble, "aleatoric": aleatoric_within,
        "epistemic": epistemic_total, "confidence": confidence,
    }


def ensemble_predict_evidential_meinert(models: List[tf.keras.Model], x, output_keys: dict) -> Dict[str, np.ndarray]:
    """نفس ensemble_predict_evidential أعلاه، لكن بعدم يقين كل نموذج مُعرَّفًا بطريقة Meinert et al. 2023
    (aleatoric=عرض Student-t w_St، epistemic=1/√ν) بدل تعريف Amini et al. 2020 الأصلي. جرّب كلا الدالتين
    وقارن المعايرة الفعلية (calibration) على بيانات تحقق حقيقية — لا يوجد تعريف "صحيح" مطلق للفصل بين
    aleatoric/epistemic، والورقتان تختلفان في أيّهما أفضل تجريبيًا حسب طبيعة البيانات."""
    mus, alea, epi_inv_sqrt_nu = [], [], []
    for m in models:
        out = m(x, training=False)
        nu = out[output_keys["nu"]].numpy()
        alpha = out[output_keys["alpha"]].numpy()
        beta = out[output_keys["beta"]].numpy()
        u = nig_uncertainties_meinert(tf.constant(nu), tf.constant(alpha), tf.constant(beta))
        mus.append(out[output_keys["mu"]].numpy())
        alea.append(u["aleatoric_wst"].numpy())
        epi_inv_sqrt_nu.append(u["epistemic_inv_sqrt_nu"].numpy())

    mus, alea, epi_inv_sqrt_nu = (np.stack(a, axis=0) for a in (mus, alea, epi_inv_sqrt_nu))
    mu_ensemble = mus.mean(axis=0)
    aleatoric_within = alea.mean(axis=0)                 # بوحدات y مباشرة (عكس aleatoric الأصلي)
    epistemic_within = epi_inv_sqrt_nu.mean(axis=0)       # بلا وحدات — مؤشر نسبي فقط، لا تقارنه رقميًا بـ aleatoric
    epistemic_between = mus.var(axis=0)

    return {
        "mu": mu_ensemble, "aleatoric": aleatoric_within,
        "epistemic_relative": epistemic_within + epistemic_between,  # لا نجمعه مع aleatoric (وحدات مختلفة)
    }


## 14) خاتمة: كيف توسّع الإطار (بدون تعديل الكود الأساسي)

| تريد إضافة/تعديل | كيف |
|---|---|
| هدف جديد بنوع موجود | أضف مفتاحًا جديدًا داخل `config['targets']` |
| نوع مهمة جديد بالكامل | `register_task_type("اسمك", دالتك, ["إحصائياتك"])` مرة واحدة، ثم استخدمه في `task_type` |
| معامل مجدول جديد (مثل penalty جديد) | أضف سطرًا في `config['loss']['schedules']`، استخدم `scheduled_vars["اسمه"]` داخل دالة الخسارة/القيد |
| قيد منطقي/فيزيائي جديد بين المخرجات | أضف `{"name":.., "fn":.., "weight_var":..}` في `config['loss']['constraints']` |
| تعطيل الموازنة التلقائية بين المهام | `config['loss']['use_uncertainty_weighting'] = False` (فعّل `task_weight_update_frequency` بدلًا منها إن رغبت) |
| تغيير جدولة معدل التعلّم | عدّل `config['optimizer']['lr_schedule']` (`constant` / `cosine` / `cosine_restarts`) |
| حذف كولباك بالكامل | لا تُضِف شرطها في `build_training_system` (أو علّق سطر `callbacks.append(...)` المقابل) |
| كولباك جديدة قابلة للاستئناف | نفّذ `get_state()` / `load_state(state)` عليها وأضفها إلى `stateful_callbacks` في `build_training_system` |
| إكمال تدريب نموذج غير مُدار بهذا الإطار | `config["run"]["train_mode"] = "warm_start"` + `config["run"]["warm_start"]` (انظر القسم 11.1) |
| البدء من الصفر عمدًا مع الاحتفاظ بالحالة القديمة | `config["run"]["train_mode"] = "new"` (تُؤرشَف الحالة القديمة في `run_dir/_archive/`، لا تُحذف) |
| اختيار «الأفضل» بأوزان EMA بدل الخام | `config["callbacks"]["early_stopping"]["weights_snapshot"] = "ema_weights"` مع `optimizer["use_ema"]=True` |

### حدود معروفة (بتصميم مقصود، لتبسيط الموثوقية)
- **الاستئناف على مستوى الحقبة (epoch) فقط**، لا على مستوى الدُفعة (batch)
  داخل الحقبة — إن انقطع التدريب في منتصف حقبة، ستُعاد تلك الحقبة كاملة من
  بدايتها عند الاستئناف. هذا يطابق سلوك أغلب أطر التدريب القياسية ويجنّبك
  تعقيد استعادة حالة `tf.data` (shuffle buffers، إلخ) بدقة على مستوى الدُفعة.
- يجب أن يكون `model_builder_fn` **مُحدِّدًا (deterministic في البنية)**: نفس
  البنية تمامًا في كل استدعاء (نفس الطبقات، نفس الأسماء، نفس الأشكال) — وإلا
  فشل `checkpoint.restore(...)` بوضوح عبر `assert_existing_objects_matched()`.
- تغيير `config['targets']` (حذف/إضافة هدف) بين التدريب الأصلي والاستئناف
  **غير مدعوم** — يتطلب ذلك بناء تشغيل جديد (`run_dir` جديد) بدل الاستئناف.